In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm import HyperbolicLCM


@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Easy",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/arc_easy_ablation"
    cache_dir: str = "arc_cached_features"

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # Training
    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: AblationConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape

        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:

        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape

        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\nAblation: {ablation_name}")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)

                global_step += 1

            current_lr = opt.param_groups[0]["lr"]

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{current_lr:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


def run_arc_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n Dataset: {dataset_name}")

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_arc(cfg, dataset_name)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    test_rows = (
        build_or_load_cached_split(
            cfg,
            dataset_name,
            "test",
            test_hf,
            conceptizer,
        )
        if test_hf is not None
        else []
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\nFINAL ABLATION TABLE")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_arc_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: ARC-Easy ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building ARC-Easy / train


cache:ARC-Easy:train: 100%|█████████████████████████████████████| 2251/2251 [04:10<00:00,  8.98it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=1.3856 train_acc=0.2599 eval_loss=1.3856 eval_acc=0.3070
[hlcm_frozen][FINAL] best_eval_acc=0.3123 final_eval_acc=0.3123 final_test_acc=0.3119 time=7.79 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=1.3832 eval_acc=0.2246


hlcm_last2_unfrozen epoch 1/2: 100%|█| 1126/1126 [10:05<00:00,  1.86it/s, acc=0.2506, loss=1.3858, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1.3858 train_acc=0.2506 eval_loss=1.3852 eval_acc=0.3386
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 1126/1126 [10:06<00:00,  1.86it/s, acc=0.2661, loss=1.3856, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=1.3856 train_acc=0.2661 eval_loss=1.3853 eval_acc=0.3526
[hlcm_last2_unfrozen] saved best.pt
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.3526 final_eval_acc=0.3526 final_test_acc=0.3338 time=22.42 min

[summary] saved runs/arc_easy_ablation/ARC-Easy/ablation_summary.csv
[summary] saved runs/arc_easy_ablation/ARC-Easy/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.3632 | test_acc=0.3400673400673401 | time=0.10 min
           hlcm_frozen | eval_acc=0.3123 | test_acc=0.31186868686868685 | time=7.79 min
   hlcm_last2_unfrozen | eval_acc=0.3526 | test_acc=0.33375420875420875 | time=22.42 min

All done.
Outputs in: runs/arc_easy_ablation
Total wall time: 47.35 min


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("ARC-Challenge",)
    arc_dataset_name: str = "allenai/ai2_arc"

    out_dir: str = "runs/arc_challenge_ablation"
    cache_dir: str = "arc_challenge_cached_features"

    # Ablations to run
    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    # pretrained H-LCM
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # DeBERTa conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # H-LCM arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # Training
    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # Ablation details
    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    # Misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# ARC DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_arc(cfg: AblationConfig, subset_name: str):
    raw = load_dataset(cfg.arc_dataset_name, subset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_arc_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])
    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_arc_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    """
    x:        [B, T, D]
    pad_mask: [B, T], True = padded
    """
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        q_emb:       [B, D]
        c_emb:       [B, K, D]
        choice_mask: [B, K]
        """
        B, K, D = c_emb.shape

        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    """
    Frozen DeBERTa concept embeddings + linear/MLP MCQ classifier.
    No H-LCM, no hyperbolic projection, no transformer reasoning.
    """
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    """
    H-LCM backbone + trainable MCQ head.

    This supports:
    - frozen H-LCM backbone
    - partially unfrozen H-LCM backbone
    """
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        """
        x:        [B, T, 768]
        pad_mask: [B, T]
        returns: [B, model_dim]
        """
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape

        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)

                global_step += 1

            current_lr = opt.param_groups[0]["lr"]

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{current_lr:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN DATASET
# ============================================================

def run_arc_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    # --------------------------------------------------------
    # Cache / load features
    # --------------------------------------------------------
    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_arc(cfg, dataset_name)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    test_rows = (
        build_or_load_cached_split(
            cfg,
            dataset_name,
            "test",
            test_hf,
            conceptizer,
        )
        if test_hf is not None
        else []
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    # --------------------------------------------------------
    # Normalizer
    # --------------------------------------------------------
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    # --------------------------------------------------------
    # Combined summary
    # --------------------------------------------------------
    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_arc_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: ARC-Challenge ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building ARC-Challenge / train


cache:ARC-Challenge:train: 100%|████████████████████████████████| 1119/1119 [02:25<00:00,  7.68it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=1.3861 train_acc=0.2627 eval_loss=1.3825 eval_acc=0.2843
[hlcm_frozen][FINAL] best_eval_acc=0.2843 final_eval_acc=0.2843 final_test_acc=0.2406 time=4.76 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=1.3808 eval_acc=0.2174


hlcm_last2_unfrozen epoch 1/2: 100%|█| 560/560 [05:11<00:00,  1.80it/s, acc=0.2565, loss=1.3862, lr=


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1.3862 train_acc=0.2565 eval_loss=1.3831 eval_acc=0.2742
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 560/560 [05:15<00:00,  1.77it/s, acc=0.2475, loss=1.3862, lr=


[hlcm_last2_unfrozen][epoch 2/2] train_loss=1.3862 train_acc=0.2475 eval_loss=1.3832 eval_acc=0.2843
[hlcm_last2_unfrozen] saved best.pt
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.2843 final_eval_acc=0.2843 final_test_acc=0.2713 time=12.41 min

[summary] saved runs/arc_challenge_ablation/ARC-Challenge/ablation_summary.csv
[summary] saved runs/arc_challenge_ablation/ARC-Challenge/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.2475 | test_acc=0.2713310580204778 | time=0.05 min
           hlcm_frozen | eval_acc=0.2843 | test_acc=0.24061433447098976 | time=4.76 min
   hlcm_last2_unfrozen | eval_acc=0.2843 | test_acc=0.2713310580204778 | time=12.41 min

All done.
Outputs in: runs/arc_challenge_ablation
Total wall time: 27.71 min


In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("OpenBookQA",)

    openbookqa_dataset_name: str = "allenai/openbookqa"
    openbookqa_config_name: str = "main"

    out_dir: str = "runs/openbookqa_ablation"
    cache_dir: str = "openbookqa_cached_features"

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# OPENBOOKQA DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_openbookqa(cfg: AblationConfig):
    raw = load_dataset(cfg.openbookqa_dataset_name, cfg.openbookqa_config_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["test"]
    test_split = raw["test"] if "test" in raw else None

    return train_split, eval_split, test_split


def normalize_openbookqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question_stem", "")

    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])

    answer_key = ex.get("answerKey", "A")

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_openbookqa_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN OPENBOOKQA
# ============================================================

def run_openbookqa_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_openbookqa(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    test_rows = (
        build_or_load_cached_split(
            cfg,
            dataset_name,
            "test",
            test_hf,
            conceptizer,
        )
        if test_hf is not None
        else []
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    if test_ds is not None:
        test_loader = DataLoader(
            test_ds,
            batch_size=cfg.eval_batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=(device.type == "cuda"),
            collate_fn=cached_collate,
            drop_last=False,
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_openbookqa_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: OpenBookQA ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building OpenBookQA / train


cache:OpenBookQA:train: 100%|███████████████████████████████████| 4957/4957 [09:03<00:00,  9.12it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=1.3858 train_acc=0.2669 eval_loss=1.3854 eval_acc=0.3160
[hlcm_frozen] saved best.pt
[hlcm_frozen][FINAL] best_eval_acc=0.3160 final_eval_acc=0.3160 final_test_acc=0.2880 time=16.04 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=1.3831 eval_acc=0.2260


hlcm_last2_unfrozen epoch 1/2: 100%|█| 2479/2479 [22:31<00:00,  1.83it/s, acc=0.2499, loss=1.3861, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1.3861 train_acc=0.2499 eval_loss=1.3853 eval_acc=0.3300
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 2479/2479 [22:19<00:00,  1.85it/s, acc=0.2736, loss=1.3853, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=1.3853 train_acc=0.2736 eval_loss=1.3850 eval_acc=0.3300
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.3300 final_eval_acc=0.3300 final_test_acc=0.3120 time=46.61 min

[summary] saved runs/openbookqa_ablation/OpenBookQA/ablation_summary.csv
[summary] saved runs/openbookqa_ablation/OpenBookQA/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.3500 | test_acc=0.318 | time=0.20 min
           hlcm_frozen | eval_acc=0.3160 | test_acc=0.288 | time=16.04 min
   hlcm_last2_unfrozen | eval_acc=0.3300 | test_acc=0.312 | time=46.61 min

All done.
Outputs in: runs/openbookqa_ablation
Total wall time: 78.57 min


In [4]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("CommonSenseQA",)

    csqa_dataset_name: str = "tau/commonsense_qa"

    out_dir: str = "runs/commonsenseqa_ablation"
    cache_dir: str = "commonsenseqa_cached_features"

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# COMMONSENSEQA DATA
# ============================================================

def answerkey_to_index(answer_key: str, labels: List[str]) -> int:
    if not isinstance(answer_key, str):
        return 0

    key = answer_key.strip()

    try:
        return labels.index(key)
    except ValueError:
        return 0


def load_commonsenseqa(cfg: AblationConfig):
    raw = load_dataset(cfg.csqa_dataset_name)

    train_split = raw["train"]
    eval_split = raw["validation"] if "validation" in raw else raw["train"]

    # CommonSenseQA test split usually has no labels, so we skip test accuracy.
    test_split = None

    return train_split, eval_split, test_split


def normalize_commonsenseqa_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")

    choices = ex.get("choices", {})
    choice_texts = choices.get("text", [])
    choice_labels = choices.get("label", [])

    answer_key = ex.get("answerKey", None)

    clean_choices = []
    clean_labels = []

    for txt, lab in zip(choice_texts, choice_labels):
        if isinstance(txt, str) and txt.strip():
            clean_choices.append(txt.strip())
            clean_labels.append(str(lab).strip())

    if len(clean_choices) < min_valid_choices:
        clean_choices = ["fallback choice A", "fallback choice B"]
        clean_labels = ["A", "B"]

    if answer_key is None or not isinstance(answer_key, str) or not answer_key.strip():
        raise ValueError("Example has no answerKey; cannot use it for supervised evaluation/training.")

    y = answerkey_to_index(answer_key, clean_labels)
    y = max(0, min(y, len(clean_choices) - 1))

    return str(q), clean_choices, y


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_commonsenseqa_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN COMMONSENSEQA
# ============================================================

def run_commonsenseqa_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_commonsenseqa(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    test_rows = []

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(test_rows) if len(test_rows) > 0 else None

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_commonsenseqa_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: CommonSenseQA ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[cache] building CommonSenseQA / train


cache:CommonSenseQA:train: 100%|████████████████████████████████| 9741/9741 [21:33<00:00,  7.53it/s]


...

hlcm_frozen epoch 2/2: 100%|█| 4871/4871 [13:40<00:00,  5.94it/s, acc=0.2286, loss=1.6075, lr=0.00e+


[hlcm_frozen][epoch 2/2] train_loss=1.6075 train_acc=0.2286 eval_loss=1.6048 eval_acc=0.2875
[hlcm_frozen][FINAL] best_eval_acc=0.2875 final_eval_acc=0.2875 time=30.17 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=1.6094 eval_acc=0.1892


hlcm_last2_unfrozen epoch 1/2: 100%|█| 4871/4871 [39:18<00:00,  2.07it/s, acc=0.2204, loss=1.6085, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1.6085 train_acc=0.2204 eval_loss=1.6043 eval_acc=0.2867
[hlcm_last2_unfrozen] saved best.pt


RuntimeError: [enforce fail at inline_container.cc:659] . unexpected pos 3771715648 vs 3771715600

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("MMLU_auxiliary_train",)

    mmlu_dataset_name: str = "cais/mmlu"
    mmlu_config_name: str = "all"

    out_dir: str = "runs/mmlu_auxiliary_train_ablation"
    cache_dir: str = "mmlu_auxiliary_train_cached_features"

    # Set True only if your previous cache was empty/broken
    rebuild_cache: bool = False

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    validation_fraction: float = 0.10

    # For fast ablation. Set both to None for full auxiliary_train.
    max_train_examples: Optional[int] = 5000
    max_eval_examples: Optional[int] = 1000

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# MMLU AUXILIARY TRAIN DATA
# ============================================================

def load_mmlu_auxiliary_train(cfg: AblationConfig):
    raw = load_dataset(cfg.mmlu_dataset_name, cfg.mmlu_config_name)

    print("[MMLU] available splits:", list(raw.keys()))

    if "auxiliary_train" not in raw:
        raise ValueError(
            f"Expected split 'auxiliary_train', but available splits are: {list(raw.keys())}"
        )

    split = raw["auxiliary_train"]

    print("[MMLU] auxiliary_train size:", len(split))
    print("[MMLU] columns:", split.column_names)
    print("[MMLU] first example:", split[0])

    split = split.shuffle(seed=cfg.seed)

    split_dict = split.train_test_split(
        test_size=cfg.validation_fraction,
        seed=cfg.seed,
    )

    train_split = split_dict["train"]
    eval_split = split_dict["test"]
    test_split = None

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        eval_split = eval_split.select(range(min(cfg.max_eval_examples, len(eval_split))))

    print("[MMLU] train size:", len(train_split))
    print("[MMLU] eval size:", len(eval_split))

    return train_split, eval_split, test_split


def normalize_mmlu_example(ex: Dict[str, Any], min_valid_choices: int):
    q = ex.get("question", "")
    choices = ex.get("choices", [])
    answer = ex.get("answer", None)

    clean_choices = []

    if isinstance(choices, list):
        for c in choices:
            if isinstance(c, str) and c.strip():
                clean_choices.append(c.strip())

    if len(clean_choices) < min_valid_choices:
        raise ValueError(f"Too few valid choices: {len(clean_choices)}")

    if answer is None:
        raise ValueError("MMLU example has no answer field.")

    y = int(answer)

    if y < 0 or y >= len(clean_choices):
        raise ValueError(f"Answer index {y} out of range for {len(clean_choices)} choices.")

    return str(q), clean_choices, y


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")

        if len(rows) == 0:
            print("[cache] cached file is empty; rebuilding.")
            os.remove(path)
        else:
            return rows

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_mmlu_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"Question: {q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError(
            "Cache building produced 0 rows. Check MMLU field names and printed first example above."
        )

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN MMLU
# ============================================================

def run_mmlu_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_mmlu_auxiliary_train(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("MMLU train dataset is empty. Check normalization/cache building.")

    if len(eval_ds) == 0:
        raise ValueError("MMLU eval dataset is empty. Check normalization/cache building.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset:", cfg.mmlu_dataset_name, cfg.mmlu_config_name)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )
    print(
        f"max_train_examples={cfg.max_train_examples}, "
        f"max_eval_examples={cfg.max_eval_examples}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_mmlu_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: cais/mmlu all
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2
max_train_examples=5000, max_eval_examples=1000

==================== Dataset: MMLU_auxiliary_train ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Using the latest cached version of the dataset since

[MMLU] available splits: ['test', 'validation', 'dev', 'auxiliary_train']
[MMLU] auxiliary_train size: 99842
[MMLU] columns: ['question', 'subject', 'choices', 'answer']
[MMLU] first example: {'question': "Davis decided to kill Adams. He set out for Adams's house. Before he got there he saw Brooks, who resembled Adams. Thinking that Brooks was Adams, Davis shot at Brooks. The shot missed Brooks but wounded Case, who was some distance away. Davis had not seen Case. In a prosecution under a statute that proscribes any attempt to commit murder, the district attorney should indicate that the intended victim(s) was/were", 'subject': '', 'choices': ['Adams only.', 'Brooks only.', 'Case only.', 'Adams and Brooks'], 'answer': 1}
[MMLU] train size: 5000
[MMLU] eval size: 1000
[cache] loading mmlu_auxiliary_train_cached_features/MMLU_auxiliary_train_train_tok256_seq8_train5000_eval1000.pt
[cache] loaded rows: 0
[cache] cached file is empty; rebuilding.
[cache] building MMLU_auxiliary_train / tra

cache:MMLU_auxiliary_train:train: 100%|█████████████████████████| 5000/5000 [31:09<00:00,  2.67it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=1.3864 train_acc=0.2454 eval_loss=1.3849 eval_acc=0.2640
[hlcm_frozen][FINAL] best_eval_acc=0.2940 final_eval_acc=0.2940 time=15.36 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=1.3829 eval_acc=0.2510


hlcm_last2_unfrozen epoch 1/2: 100%|█| 2500/2500 [21:58<00:00,  1.90it/s, acc=0.2318, loss=1.3865, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1.3865 train_acc=0.2318 eval_loss=1.3849 eval_acc=0.2560
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 2500/2500 [21:49<00:00,  1.91it/s, acc=0.2418, loss=1.3863, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=1.3863 train_acc=0.2418 eval_loss=1.3851 eval_acc=0.2520
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.2560 final_eval_acc=0.2560 time=46.21 min

[summary] saved runs/mmlu_auxiliary_train_ablation/MMLU_auxiliary_train/ablation_summary.csv
[summary] saved runs/mmlu_auxiliary_train_ablation/MMLU_auxiliary_train/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.3100 | test_acc=NA | time=0.20 min
           hlcm_frozen | eval_acc=0.2940 | test_acc=NA | time=15.36 min
   hlcm_last2_unfrozen | eval_acc=0.2560 | test_acc=NA | time=46.21 min

All done.
Outputs in: runs/mmlu_auxiliary_train_ablation
Total wall time: 104.91 min


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("BoolQ",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet"

    out_dir: str = "runs/boolq_ablation"
    cache_dir: str = "boolq_cached_features"

    rebuild_cache: bool = False

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    # Set to None to use full train/validation files.
    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# BOOLQ DATA
# ============================================================

def load_boolq(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"BoolQ train file not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"BoolQ validation file not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    train_split = raw["train"]
    eval_split = raw["validation"]
    test_split = None

    train_split = train_split.shuffle(seed=cfg.seed)

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        eval_split = eval_split.select(range(min(cfg.max_eval_examples, len(eval_split))))

    print("[BoolQ] train size:", len(train_split))
    print("[BoolQ] validation size:", len(eval_split))
    print("[BoolQ] columns:", train_split.column_names)
    print("[BoolQ] first example:", train_split[0])

    return train_split, eval_split, test_split


def normalize_boolq_example(ex: Dict[str, Any]):
    question = ex.get("question", "")
    passage = ex.get("passage", "")
    answer = ex.get("answer", None)

    if answer is None:
        raise ValueError("BoolQ example has no answer field.")

    # BoolQ answer is boolean.
    label = 1 if bool(answer) else 0

    q_text = (
        f"Passage: {str(passage)}\n"
        f"Question: {str(question)}\n"
        f"Is the answer true or false?"
    )

    choice_texts = ["false", "true"]

    return q_text, choice_texts, label


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")

        if len(rows) == 0:
            print("[cache] cached file is empty; rebuilding.")
            os.remove(path)
        else:
            return rows

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_boolq_example(ex)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"{q_text}\nCandidate Answer: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check BoolQ parquet fields.")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN BOOLQ
# ============================================================

def run_boolq_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf, test_hf = load_boolq(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("BoolQ train dataset is empty. Check normalization/cache building.")

    if len(eval_ds) == 0:
        raise ValueError("BoolQ validation dataset is empty. Check normalization/cache building.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: BoolQ")
    print("Train file:", cfg.hf_train_file)
    print("Validation file:", cfg.hf_validation_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_boolq_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: BoolQ
Train file: /home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet
Validation file: /home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: BoolQ ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BoolQ] train size: 9427
[BoolQ] validation size: 3270
[BoolQ] columns: ['question', 'answer', 'passage']
[BoolQ] first example: {'question': 'did henry die in once upon a time', 'answer': False, 'passage': "Henry Daniel Mills is a fictional character in ABC's television series Once Upon a Time. Henry is the boy Emma Swan gave up to adoption; Regina Mills adopted him. Henry was originally portrayed as a child by Jared S. Gilmore, who won the Young Artist Award for Best Performance in a TV Series -- Leading Young Actor in 2012. For the show's seventh and final season, Andrew J. West later took over the role of Henry as an adult and father to a eight-year-old girl named Lucy, with Gilmore also making three appearances as Henry during the season."}
[cache] building BoolQ / train


cache:BoolQ:train: 100%|████████████████████████████████████████| 9427/9427 [18:13<00:00,  8.62it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=0.6928 train_acc=0.5039 eval_loss=0.6921 eval_acc=0.4376
[hlcm_frozen][FINAL] best_eval_acc=0.4960 final_eval_acc=0.4960 time=28.92 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=0.6914 eval_acc=0.4575


hlcm_last2_unfrozen epoch 1/2: 100%|█| 4714/4714 [40:08<00:00,  1.96it/s, acc=0.4912, loss=0.6932, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=0.6932 train_acc=0.4912 eval_loss=0.6922 eval_acc=0.4765
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 4714/4714 [40:06<00:00,  1.96it/s, acc=0.5026, loss=0.6931, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=0.6931 train_acc=0.5026 eval_loss=0.6923 eval_acc=0.4865
[hlcm_last2_unfrozen] saved best.pt
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.4865 final_eval_acc=0.4865 time=85.97 min

[summary] saved runs/boolq_ablation/BoolQ/ablation_summary.csv
[summary] saved runs/boolq_ablation/BoolQ/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.6159 | test_acc=NA | time=0.35 min
           hlcm_frozen | eval_acc=0.4960 | test_acc=NA | time=28.92 min
   hlcm_last2_unfrozen | eval_acc=0.4865 | test_acc=NA | time=85.97 min

All done.
Outputs in: runs/boolq_ablation
Total wall time: 150.79 min


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("MultiRC",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/multirc/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/multirc/validation-00000-of-00001.parquet"

    out_dir: str = "runs/multirc_ablation_8k"
    cache_dir: str = "multirc_cached_features_8k"

    # Set True so old full-data cache is not reused
    rebuild_cache: bool = True

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    # Only 8000 training samples
    max_train_examples: Optional[int] = 8000

    # Use full validation set
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0

# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# MULTIRC DATA
# ============================================================

def load_multirc(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"MultiRC train file not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"MultiRC validation file not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    train_split = raw["train"]
    eval_split = raw["validation"]
    test_split = None

    train_split = train_split.shuffle(seed=cfg.seed)

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        eval_split = eval_split.select(range(min(cfg.max_eval_examples, len(eval_split))))

    print("[MultiRC] train size:", len(train_split))
    print("[MultiRC] validation size:", len(eval_split))
    print("[MultiRC] columns:", train_split.column_names)
    print("[MultiRC] first example:", train_split[0])

    return train_split, eval_split, test_split


def normalize_multirc_example(ex: Dict[str, Any]):
    paragraph = ex.get("paragraph", "")
    question = ex.get("question", "")
    answer = ex.get("answer", "")
    label = ex.get("label", None)

    if label is None:
        raise ValueError("MultiRC example has no label field.")

    label = int(label)

    if label not in [0, 1]:
        raise ValueError(f"Invalid MultiRC label: {label}")

    q_text = (
        f"Paragraph: {str(paragraph)}\n"
        f"Question: {str(question)}\n"
        f"Candidate Answer: {str(answer)}\n"
        f"Is this candidate answer correct?"
    )

    choice_texts = ["false", "true"]

    return q_text, choice_texts, label


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")

    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")

        if len(rows) == 0:
            print("[cache] cached file is empty; rebuilding.")
            os.remove(path)
        else:
            return rows

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_multirc_example(ex)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"{q_text}\nCandidate Label: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)

    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check MultiRC parquet fields.")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)

        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(
        self,
        q_emb: torch.Tensor,
        c_emb: torch.Tensor,
        choice_mask: torch.Tensor,
    ) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat(
            [
                q,
                c_emb,
                torch.abs(q - c_emb),
                q * c_emb,
            ],
            dim=-1,
        )

        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)

        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        logits = self.head(q_emb, c_emb, choice_mask)

        return logits


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    logits = model(batch)
    labels = batch["label"].to(device)

    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()

    return loss, acc


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)

        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]

    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(
        param_groups,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN MULTIRC
# ============================================================

def run_multirc_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_multirc(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "validation",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("MultiRC train dataset is empty. Check normalization/cache building.")

    if len(eval_ds) == 0:
        raise ValueError("MultiRC validation dataset is empty. Check normalization/cache building.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = None

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: MultiRC")
    print("Train file:", cfg.hf_train_file)
    print("Validation file:", cfg.hf_validation_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_multirc_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: MultiRC
Train file: /home/user/twovolume/Nisha/Finetune/multirc/train-00000-of-00001.parquet
Validation file: /home/user/twovolume/Nisha/Finetune/multirc/validation-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: MultiRC ====================
[cache] rebuild_cache=True. Removing cache directory: multirc_cached_features_8k


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MultiRC] train size: 8000
[MultiRC] validation size: 4848
[MultiRC] columns: ['paragraph', 'question', 'answer', 'idx', 'label']
[MultiRC] first example: {'paragraph': "As for the Italians, we know that Paesiello, who was a famous intriguer against his musical rivals, was a devoted husband whose wife was an invalid and who died soon after her death. Cherubini married Mademoiselle Cecile Turette, when he was thirty-five, and the marriage was not a success. He left a son and two daughters. Spontini, one of whose best operas was based on the life of that much mis-married enthusiast for divorce, John Milton, took to wife a member of the Erard family. In the outer world Spontini was famous for his despotism, his jealousy, his bad temper, and his excessive vanity. None of these qualities as a rule add much to home comfort, and yet, it is said that he lived happily with his wife. We may feel sure that some of the bad light thrown on his character is due purely to the jealousy of rivals, when

cache:MultiRC:train: 100%|██████████████████████████████████████| 8000/8000 [30:42<00:00,  4.34it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=0.6932 train_acc=0.4969 eval_loss=0.6921 eval_acc=0.5588
[hlcm_frozen] saved best.pt
[hlcm_frozen][FINAL] best_eval_acc=0.5588 final_eval_acc=0.5588 time=27.86 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=0.6914 eval_acc=0.5344


hlcm_last2_unfrozen epoch 1/2: 100%|█| 4000/4000 [31:32<00:00,  2.11it/s, acc=0.4930, loss=0.6932, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=0.6932 train_acc=0.4930 eval_loss=0.6921 eval_acc=0.5631
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 4000/4000 [31:40<00:00,  2.10it/s, acc=0.5045, loss=0.6930, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=0.6930 train_acc=0.5045 eval_loss=0.6921 eval_acc=0.5594
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.5631 final_eval_acc=0.5631 time=70.26 min

[summary] saved runs/multirc_ablation_8k/MultiRC/ablation_summary.csv
[summary] saved runs/multirc_ablation_8k/MultiRC/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.5359 | test_acc=NA | time=0.33 min
           hlcm_frozen | eval_acc=0.5588 | test_acc=NA | time=27.86 min
   hlcm_last2_unfrozen | eval_acc=0.5631 | test_acc=NA | time=70.26 min

All done.
Outputs in: runs/multirc_ablation_8k
Total wall time: 163.15 min


In [2]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("MAWPS",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/mawps_regression_ablation"
    cache_dir: str = "mawps_regression_cached_features"

    rebuild_cache: bool = False

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# MAWPS DATA
# ============================================================

def load_mawps(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"MAWPS train file not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_test_file):
        raise FileNotFoundError(f"MAWPS test file not found: {cfg.hf_test_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"].shuffle(seed=cfg.seed)
    test_split = raw["test"]

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        test_split = test_split.select(range(min(cfg.max_eval_examples, len(test_split))))

    print("[MAWPS] train size:", len(train_split))
    print("[MAWPS] test size:", len(test_split))
    print("[MAWPS] columns:", train_split.column_names)
    print("[MAWPS] first example:", train_split[0])

    return train_split, test_split, test_split


def extract_final_numeric_answer(answer_text: str) -> float:
    text = str(answer_text)

    if "####" in text:
        final_part = text.split("####")[-1].strip()
    else:
        final_part = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final_part)
    if nums:
        return float(nums[-1])

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return float(nums[-1])

    raise ValueError(f"Could not extract numeric answer from: {answer_text}")


def normalize_mawps_example(ex: Dict[str, Any]):
    question = ex.get("question", "")
    answer_text = ex.get("answer", "")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("MAWPS example has empty question.")

    y = extract_final_numeric_answer(answer_text)

    input_text = (
        f"Math word problem: {question}\n"
        f"Solve the problem and predict the final numeric answer."
    )

    return input_text, float(y)


# ============================================================
# FEATURE CACHE FOR REGRESSION
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_regression_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")
        if len(rows) > 0:
            return rows
        os.remove(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            text, label = normalize_mawps_example(ex)
            seq, pad = conceptizer.encode_text_fixed(text)

            rows.append({
                "x": seq,
                "mask": pad,
                "label": float(label),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check MAWPS fields.")

    return rows


class CachedRegressionDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "mask": r["mask"],
            "label": torch.tensor(r["label"], dtype=torch.float32),
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    x = torch.stack([item["x"] for item in batch], dim=0)
    mask = torch.stack([item["mask"] for item in batch], dim=0)
    labels = torch.stack([item["label"] for item in batch], dim=0)
    idxs = torch.tensor([item["idx"] for item in batch], dtype=torch.long)

    return {
        "x": x,
        "mask": mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# REGRESSION MODELS
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class RegressionHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, emb: torch.Tensor) -> torch.Tensor:
        return self.net(emb).squeeze(-1)


class DebertaOnlyRegressionModel(nn.Module):
    """
    Frozen DeBERTa concept embeddings + regression head.
    No H-LCM, no hyperbolic reasoning.
    """
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = RegressionHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        x = batch["x"].to(device)
        mask = batch["mask"].to(device)

        emb = masked_mean_pool(x, mask)
        pred = self.head(emb)

        return pred


class HLCMRegressionAblationModel(nn.Module):
    """
    H-LCM backbone + numeric regression head.
    """
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = RegressionHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = batch["x"]
        mask = batch["mask"]

        emb = self.encode_hlcm(x, mask)
        emb = F.normalize(emb, dim=-1)

        pred = self.head(emb)

        return pred


def freeze_hlcm_backbone(model: HLCMRegressionAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMRegressionAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL FOR REGRESSION
# ============================================================

def regression_metrics(pred: torch.Tensor, y: torch.Tensor) -> Dict[str, float]:
    pred = pred.detach().float()
    y = y.detach().float()

    mae = torch.mean(torch.abs(pred - y)).item()
    rmse = torch.sqrt(torch.mean((pred - y) ** 2)).item()

    rounded_acc = (torch.round(pred) == torch.round(y)).float().mean().item()

    return {
        "mae": mae,
        "rmse": rmse,
        "rounded_acc": rounded_acc,
    }


def compute_loss_metrics(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    pred = model(batch)
    y = batch["label"].to(device)

    loss = F.smooth_l1_loss(pred, y)
    metrics = regression_metrics(pred, y)

    return loss, metrics


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "mae": 0.0, "rmse": 0.0, "rounded_acc": 0.0}

    model.eval()

    total_loss = 0.0
    total_abs = 0.0
    total_sq = 0.0
    total_exact = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                pred = model(batch)
                y = batch["label"].to(device)
                loss = F.smooth_l1_loss(pred, y)
        else:
            pred = model(batch)
            y = batch["label"].to(device)
            loss = F.smooth_l1_loss(pred, y)

        bs = y.size(0)

        total_loss += float(loss.item()) * bs
        total_abs += torch.sum(torch.abs(pred.float() - y.float())).item()
        total_sq += torch.sum((pred.float() - y.float()) ** 2).item()
        total_exact += torch.sum(torch.round(pred.float()) == torch.round(y.float())).item()
        total_n += bs

    mae = total_abs / max(1, total_n)
    rmse = math.sqrt(total_sq / max(1, total_n))
    rounded_acc = total_exact / max(1, total_n)

    return {
        "loss": total_loss / max(1, total_n),
        "mae": mae,
        "rmse": rmse,
        "rounded_acc": rounded_acc,
    }


def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]
    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)

    print(
        f"[BASE] loss={base_eval['loss']:.4f} "
        f"mae={base_eval['mae']:.4f} "
        f"rmse={base_eval['rmse']:.4f} "
        f"rounded_acc={base_eval['rounded_acc']:.4f}"
    )

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_mae": "",
        "train_rmse": "",
        "train_rounded_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_mae": base_eval["mae"],
        "eval_rmse": base_eval["rmse"],
        "eval_rounded_acc": base_eval["rounded_acc"],
    })

    best_mae = base_eval["mae"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_abs_sum = 0.0
        epoch_sq_sum = 0.0
        epoch_exact_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    pred = model(batch)
                    y = batch["label"].to(device)
                    loss = F.smooth_l1_loss(pred, y)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                pred = model(batch)
                y = batch["label"].to(device)
                loss = F.smooth_l1_loss(pred, y)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = y.size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_abs_sum += torch.sum(torch.abs(pred.float() - y.float())).item()
            epoch_sq_sum += torch.sum((pred.float() - y.float()) ** 2).item()
            epoch_exact_sum += torch.sum(torch.round(pred.float()) == torch.round(y.float())).item()
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            train_loss_running = epoch_loss_sum / max(1, epoch_count)
            train_mae_running = epoch_abs_sum / max(1, epoch_count)
            train_rmse_running = math.sqrt(epoch_sq_sum / max(1, epoch_count))
            train_acc_running = epoch_exact_sum / max(1, epoch_count)

            pbar.set_postfix(
                loss=f"{train_loss_running:.4f}",
                mae=f"{train_mae_running:.4f}",
                acc=f"{train_acc_running:.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_mae = epoch_abs_sum / max(1, epoch_count)
        train_rmse = math.sqrt(epoch_sq_sum / max(1, epoch_count))
        train_rounded_acc = epoch_exact_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "train_rmse": train_rmse,
            "train_rounded_acc": train_rounded_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "train_rmse": train_rmse,
            "train_rounded_acc": train_rounded_acc,
            "eval_loss": eval_result["loss"],
            "eval_mae": eval_result["mae"],
            "eval_rmse": eval_result["rmse"],
            "eval_rounded_acc": eval_result["rounded_acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_mae={train_mae:.4f} "
            f"train_rmse={train_rmse:.4f} "
            f"train_rounded_acc={train_rounded_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_mae={eval_result['mae']:.4f} "
            f"eval_rmse={eval_result['rmse']:.4f} "
            f"eval_rounded_acc={eval_result['rounded_acc']:.4f}"
        )

        if eval_result["mae"] < best_mae:
            best_mae = eval_result["mae"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_mae": best_mae,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_mae": best_mae,
        "final_eval_loss": final_eval["loss"],
        "final_eval_mae": final_eval["mae"],
        "final_eval_rmse": final_eval["rmse"],
        "final_eval_rounded_acc": final_eval["rounded_acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_mae": None if final_test is None else final_test["mae"],
        "final_test_rmse": None if final_test is None else final_test["rmse"],
        "final_test_rounded_acc": None if final_test is None else final_test["rounded_acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"eval_mae={final_eval['mae']:.4f} "
        f"eval_rmse={final_eval['rmse']:.4f} "
        f"eval_rounded_acc={final_eval['rounded_acc']:.4f} "
        + (
            f"test_mae={final_test['mae']:.4f} "
            f"test_rmse={final_test['rmse']:.4f} "
            f"test_rounded_acc={final_test['rounded_acc']:.4f} "
            if final_test is not None else ""
        )
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN MAWPS
# ============================================================

def run_mawps_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_mawps(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "test",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedRegressionDataset(train_rows)
    eval_ds = CachedRegressionDataset(eval_rows)
    test_ds = CachedRegressionDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("MAWPS train dataset is empty.")

    if len(eval_ds) == 0:
        raise ValueError("MAWPS test dataset is empty.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = eval_loader

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_mae={row['final_eval_mae']:.4f} | "
            f"eval_rmse={row['final_eval_rmse']:.4f} | "
            f"eval_rounded_acc={row['final_eval_rounded_acc']:.4f} | "
            f"test_mae={row['final_test_mae']:.4f} | "
            f"test_rounded_acc={row['final_test_rounded_acc']:.4f} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: MAWPS")
    print("Train file:", cfg.hf_train_file)
    print("Test file:", cfg.hf_test_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_mawps_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: MAWPS
Train file: /home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet
Test file: /home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: MAWPS ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MAWPS] train size: 1417
[MAWPS] test size: 355
[MAWPS] columns: ['question', 'answer']
[MAWPS] first example: {'question': "Alyssa 's dog had puppies . She gave 7 to her friends . She now has 5 puppies . How many puppies did she have to start with ?", 'answer': '7 + 5 = 12 #### 12'}
[cache] building MAWPS / train


cache:MAWPS:train: 100%|████████████████████████████████████████| 1417/1417 [00:45<00:00, 31.20it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=1801.2250 train_mae=1801.7160 train_rmse=27795.5645 train_rounded_acc=0.0318 eval_loss=804.7756 eval_mae=805.2665 eval_rmse=7050.7110 eval_rounded_acc=0.0338
[hlcm_frozen] saved best.pt
[hlcm_frozen][FINAL] eval_mae=805.2665 eval_rmse=7050.7110 eval_rounded_acc=0.0338 test_mae=805.2665 test_rmse=7050.7110 test_rounded_acc=0.0338 time=4.37 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,436,493,314 trainable=419,545,090
[BASE] loss=808.2862 mae=808.7785 rmse=7051.2149 rounded_acc=0.0254


hlcm_last2_unfrozen epoch 1/2: 100%|█| 709/709 [02:29<00:00,  4.75it/s, acc=0.0261, loss=1801.8056, 


[hlcm_last2_unfrozen][epoch 1/2] train_loss=1801.8056 train_mae=1802.2952 train_rmse=27795.6040 train_rounded_acc=0.0261 eval_loss=804.8911 eval_mae=805.3804 eval_rmse=7050.7297 eval_rounded_acc=0.0338
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 709/709 [02:37<00:00,  4.50it/s, acc=0.0353, loss=1799.6085, 


[hlcm_last2_unfrozen][epoch 2/2] train_loss=1799.6085 train_mae=1800.0981 train_rmse=27795.4107 train_rounded_acc=0.0353 eval_loss=804.0928 eval_mae=804.5823 eval_rmse=7050.5730 eval_rounded_acc=0.0451
[hlcm_last2_unfrozen] saved best.pt
[hlcm_last2_unfrozen][FINAL] eval_mae=804.5823 eval_rmse=7050.5730 eval_rounded_acc=0.0451 test_mae=804.5823 test_rmse=7050.5730 test_rounded_acc=0.0451 time=7.32 min

[summary] saved runs/mawps_regression_ablation/MAWPS/ablation_summary.csv
[summary] saved runs/mawps_regression_ablation/MAWPS/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_mae=802.2908 | eval_rmse=7049.6597 | eval_rounded_acc=0.0169 | test_mae=802.2908 | test_rounded_acc=0.0169 | time=0.06 min
           hlcm_frozen | eval_mae=805.2665 | eval_rmse=7050.7110 | eval_rounded_acc=0.0338 | test_mae=805.2665 | test_rounded_acc=0.0338 | time=4.37 min
   hlcm_last2_unfrozen | eval_mae=804.5823 | eval_rmse=7050.5730 | eval_rounded_acc=0.0451 | te

In [3]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math, time, json, random, csv, gc, shutil, re
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("GSM8K",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    out_dir: str = "runs/gsm8k_regression_ablation"
    cache_dir: str = "gsm8k_regression_cached_features"
    rebuild_cache: bool = False

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


def load_gsm8k(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"GSM8K train file not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_test_file):
        raise FileNotFoundError(f"GSM8K test file not found: {cfg.hf_test_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"].shuffle(seed=cfg.seed)
    test_split = raw["test"]

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))
    if cfg.max_eval_examples is not None:
        test_split = test_split.select(range(min(cfg.max_eval_examples, len(test_split))))

    print("[GSM8K] train size:", len(train_split))
    print("[GSM8K] test size:", len(test_split))
    print("[GSM8K] columns:", train_split.column_names)
    print("[GSM8K] first example:", train_split[0])

    return train_split, test_split, test_split


def extract_final_numeric_answer(answer_text: str) -> float:
    text = str(answer_text).replace(",", "")

    if "####" in text:
        final_part = text.split("####")[-1].strip()
    else:
        final_part = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final_part)
    if nums:
        return float(nums[-1])

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return float(nums[-1])

    raise ValueError(f"Could not extract numeric answer from: {answer_text}")


def normalize_gsm8k_example(ex: Dict[str, Any]):
    question = ex.get("question", "")
    answer_text = ex.get("answer", "")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("GSM8K example has empty question.")

    y = extract_final_numeric_answer(answer_text)

    input_text = (
        f"Math word problem: {question}\n"
        f"Solve the problem and predict the final numeric answer."
    )

    return input_text, float(y)


def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_regression_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(cfg, dataset_name, split_name, hf_split, conceptizer):
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")
        if len(rows) > 0:
            return rows
        os.remove(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows, skipped = [], 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            text, label = normalize_gsm8k_example(ex)
            seq, pad = conceptizer.encode_text_fixed(text)
            rows.append({"x": seq, "mask": pad, "label": float(label)})
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check GSM8K fields.")

    return rows


class CachedRegressionDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "x": r["x"],
            "mask": r["mask"],
            "label": torch.tensor(r["label"], dtype=torch.float32),
            "idx": idx,
        }


def cached_collate(batch):
    x = torch.stack([item["x"] for item in batch], dim=0)
    mask = torch.stack([item["mask"] for item in batch], dim=0)
    labels = torch.stack([item["label"] for item in batch], dim=0)
    idxs = torch.tensor([item["idx"] for item in batch], dtype=torch.long)
    return {"x": x, "mask": mask, "label": labels, "idx": idxs}


def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


class RegressionHead(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, emb):
        return self.net(emb).squeeze(-1)


class HLCMRegressionAblationModel(nn.Module):
    def __init__(self, hlcm, model_dim, dropout=0.1, mu=None, sigma=None):
        super().__init__()
        self.hlcm = hlcm
        self.head = RegressionHead(dim=model_dim, dropout=dropout)
        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x, pad_mask):
        device = next(self.parameters()).device
        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch):
        emb = self.encode_hlcm(batch["x"], batch["mask"])
        emb = F.normalize(emb, dim=-1)
        return self.head(emb)


def freeze_hlcm_backbone(model):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model, n_last):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


@torch.no_grad()
def evaluate(model, loader, device, cfg):
    if loader is None:
        return {"loss": 0.0, "mae": 0.0, "rmse": 0.0, "rounded_acc": 0.0}

    model.eval()
    total_loss, total_abs, total_sq, total_exact, total_n = 0.0, 0.0, 0.0, 0.0, 0
    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                pred = model(batch)
                y = batch["label"].to(device)
                loss = F.smooth_l1_loss(pred, y)
        else:
            pred = model(batch)
            y = batch["label"].to(device)
            loss = F.smooth_l1_loss(pred, y)

        bs = y.size(0)
        total_loss += float(loss.item()) * bs
        total_abs += torch.sum(torch.abs(pred.float() - y.float())).item()
        total_sq += torch.sum((pred.float() - y.float()) ** 2).item()
        total_exact += torch.sum(torch.round(pred.float()) == torch.round(y.float())).item()
        total_n += bs

    mae = total_abs / max(1, total_n)
    rmse = math.sqrt(total_sq / max(1, total_n))
    rounded_acc = total_exact / max(1, total_n)

    return {
        "loss": total_loss / max(1, total_n),
        "mae": mae,
        "rmse": rmse,
        "rounded_acc": rounded_acc,
    }


def make_optimizer_and_scheduler(model, ablation_name, cfg, total_steps):
    if ablation_name == "hlcm_last2_unfrozen":
        param_groups = [
            {"params": [p for p in model.head.parameters() if p.requires_grad], "lr": cfg.head_lr},
            {"params": [p for p in model.hlcm.parameters() if p.requires_grad], "lr": cfg.backbone_lr},
        ]
    else:
        param_groups = [{"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr}]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


def train_one_ablation(model, ablation_name, train_loader, eval_loader, test_loader, out_dir, cfg, device):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(model, ablation_name, cfg, total_steps)
    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(
        f"[BASE] loss={base_eval['loss']:.4f} "
        f"mae={base_eval['mae']:.4f} "
        f"rmse={base_eval['rmse']:.4f} "
        f"rounded_acc={base_eval['rounded_acc']:.4f}"
    )

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_mae": "",
        "train_rmse": "",
        "train_rounded_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_mae": base_eval["mae"],
        "eval_rmse": base_eval["rmse"],
        "eval_rounded_acc": base_eval["rounded_acc"],
    })

    best_mae = base_eval["mae"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum, epoch_abs_sum, epoch_sq_sum, epoch_exact_sum, epoch_count = 0, 0, 0, 0, 0

        pbar = tqdm(train_loader, desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    pred = model(batch)
                    y = batch["label"].to(device)
                    loss = F.smooth_l1_loss(pred, y)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                pred = model(batch)
                y = batch["label"].to(device)
                loss = F.smooth_l1_loss(pred, y)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = y.size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_abs_sum += torch.sum(torch.abs(pred.float() - y.float())).item()
            epoch_sq_sum += torch.sum((pred.float() - y.float()) ** 2).item()
            epoch_exact_sum += torch.sum(torch.round(pred.float()) == torch.round(y.float())).item()
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                mae=f"{epoch_abs_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_exact_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_mae = epoch_abs_sum / max(1, epoch_count)
        train_rmse = math.sqrt(epoch_sq_sum / max(1, epoch_count))
        train_rounded_acc = epoch_exact_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "train_rmse": train_rmse,
            "train_rounded_acc": train_rounded_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_mae": train_mae,
            "train_rmse": train_rmse,
            "train_rounded_acc": train_rounded_acc,
            "eval_loss": eval_result["loss"],
            "eval_mae": eval_result["mae"],
            "eval_rmse": eval_result["rmse"],
            "eval_rounded_acc": eval_result["rounded_acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} train_mae={train_mae:.4f} "
            f"train_rmse={train_rmse:.4f} train_rounded_acc={train_rounded_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} eval_mae={eval_result['mae']:.4f} "
            f"eval_rmse={eval_result['rmse']:.4f} eval_rounded_acc={eval_result['rounded_acc']:.4f}"
        )

        if eval_result["mae"] < best_mae:
            best_mae = eval_result["mae"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_mae": best_mae,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )
            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0
    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None
    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_mae": best_mae,
        "final_eval_loss": final_eval["loss"],
        "final_eval_mae": final_eval["mae"],
        "final_eval_rmse": final_eval["rmse"],
        "final_eval_rounded_acc": final_eval["rounded_acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_mae": None if final_test is None else final_test["mae"],
        "final_test_rmse": None if final_test is None else final_test["rmse"],
        "final_test_rounded_acc": None if final_test is None else final_test["rounded_acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"eval_mae={final_eval['mae']:.4f} "
        f"eval_rmse={final_eval['rmse']:.4f} "
        f"eval_rounded_acc={final_eval['rounded_acc']:.4f} "
        + (
            f"test_mae={final_test['mae']:.4f} "
            f"test_rmse={final_test['rmse']:.4f} "
            f"test_rounded_acc={final_test['rounded_acc']:.4f} "
            if final_test is not None else ""
        )
        + f"time={total_minutes:.2f} min"
    )

    del opt, sched, scaler
    cuda_cleanup()
    return summary


def build_ablation_model(ablation_name, cfg, device, mu, sigma):
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMRegressionAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


def run_gsm8k_ablation(dataset_name, cfg, device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_gsm8k(cfg)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "test", eval_hf, conceptizer)

    del conceptizer
    cuda_cleanup()

    train_ds = CachedRegressionDataset(train_rows)
    eval_ds = CachedRegressionDataset(eval_rows)
    test_ds = CachedRegressionDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("GSM8K train dataset is empty.")
    if len(eval_ds) == 0:
        raise ValueError("GSM8K test dataset is empty.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = eval_loader
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(ablation_name, cfg, device, mu, sigma)

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")
    if all_summaries:
        fieldnames = list(all_summaries[0].keys())
        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")
    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_mae={row['final_eval_mae']:.4f} | "
            f"eval_rmse={row['final_eval_rmse']:.4f} | "
            f"eval_rounded_acc={row['final_eval_rounded_acc']:.4f} | "
            f"test_mae={row['final_test_mae']:.4f} | "
            f"test_rounded_acc={row['final_test_rounded_acc']:.4f} | "
            f"time={row['total_minutes']:.2f} min"
        )


def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: GSM8K")
    print("Train file:", cfg.hf_train_file)
    print("Test file:", cfg.hf_test_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_gsm8k_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: GSM8K
Train file: /home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet
Test file: /home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8

==================== Dataset: GSM8K ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[GSM8K] train size: 7473
[GSM8K] test size: 1319
[GSM8K] columns: ['question', 'answer']
[GSM8K] first example: {'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16'}
[cache] building GSM8K / train


cache:GSM8K:train: 100%|████████████████████████████████████████| 7473/7473 [04:12<00:00, 29.63it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=53928.4312 train_mae=53928.9277 train_rmse=2582275.9627 train_rounded_acc=0.0088 eval_loss=6814.7188 eval_mae=6815.2170 eval_rmse=92053.4246 eval_rounded_acc=0.0061
[hlcm_frozen][FINAL] eval_mae=6815.1685 eval_rmse=92054.2708 eval_rounded_acc=0.0045 test_mae=6815.1685 test_rmse=92054.2708 test_rounded_acc=0.0045 time=12.23 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,436,493,314 trainable=419,545,090
[BASE] loss=6829.8353 mae=6830.3353 rmse=92057.0159 rounded_acc=0.0000


hlcm_last2_unfrozen epoch 1/2: 100%|█| 3737/3737 [12:47<00:00,  4.87it/s, acc=0.0111, loss=53932.651


[hlcm_last2_unfrozen][epoch 1/2] train_loss=53932.6513 train_mae=53933.1473 train_rmse=2582276.3001 train_rounded_acc=0.0111 eval_loss=6814.5901 eval_mae=6815.0895 eval_rmse=92054.1296 eval_rounded_acc=0.0015
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 3737/3737 [13:50<00:00,  4.50it/s, acc=0.0045, loss=53926.315


[hlcm_last2_unfrozen][epoch 2/2] train_loss=53926.3156 train_mae=53926.8137 train_rmse=2582275.6777 train_rounded_acc=0.0045 eval_loss=6814.6008 eval_mae=6815.1005 eval_rmse=92053.6328 eval_rounded_acc=0.0000
[hlcm_last2_unfrozen][FINAL] eval_mae=6815.0895 eval_rmse=92054.1296 eval_rounded_acc=0.0015 test_mae=6815.0895 test_rmse=92054.1296 test_rounded_acc=0.0015 time=28.62 min

[summary] saved runs/gsm8k_regression_ablation/GSM8K/ablation_summary.csv
[summary] saved runs/gsm8k_regression_ablation/GSM8K/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_mae=6814.9238 | eval_rmse=92053.8076 | eval_rounded_acc=0.0030 | test_mae=6814.9238 | test_rounded_acc=0.0030 | time=0.23 min
           hlcm_frozen | eval_mae=6815.1685 | eval_rmse=92054.2708 | eval_rounded_acc=0.0045 | test_mae=6815.1685 | test_rounded_acc=0.0045 | time=12.23 min
   hlcm_last2_unfrozen | eval_mae=6815.0895 | eval_rmse=92054.1296 | eval_rounded_acc=0.0015 | test_mae=6815.089

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("MAWPS",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet"

    out_dir: str = "runs/mawps_ablation"
    cache_dir: str = "mawps_cached_features"

    rebuild_cache: bool = False

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# MAWPS DATA
# ============================================================

def load_mawps(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"MAWPS train file not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_test_file):
        raise FileNotFoundError(f"MAWPS test file not found: {cfg.hf_test_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"]
    test_split = raw["test"]

    train_split = train_split.shuffle(seed=cfg.seed)

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        test_split = test_split.select(range(min(cfg.max_eval_examples, len(test_split))))

    print("[MAWPS] train size:", len(train_split))
    print("[MAWPS] test size:", len(test_split))
    print("[MAWPS] columns:", train_split.column_names)
    print("[MAWPS] first example:", train_split[0])

    return train_split, test_split, test_split


def extract_final_numeric_answer(answer_text: str) -> str:
    text = str(answer_text)

    if "####" in text:
        final = text.split("####")[-1].strip()
    else:
        final = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final)
    if nums:
        return nums[-1]

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    raise ValueError(f"Could not extract numeric answer from: {answer_text}")


def make_wrong_numeric_answer(correct: str) -> str:
    try:
        if "." in correct:
            val = float(correct)
            wrong = val + 1.0 if val != -1.0 else val + 2.0
            return str(round(wrong, 4)).rstrip("0").rstrip(".")
        else:
            val = int(correct)
            wrong = val + 1 if val != -1 else val + 2
            return str(wrong)
    except Exception:
        return f"{correct}_wrong"


def normalize_mawps_example(ex: Dict[str, Any], seed: int = 42):
    question = ex.get("question", "")
    answer_text = ex.get("answer", "")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("MAWPS example has empty question.")

    correct = extract_final_numeric_answer(answer_text)
    wrong = make_wrong_numeric_answer(correct)

    # Deterministic shuffle per example.
    key = str(question) + str(answer_text)
    local_seed = abs(hash(key)) % (2 ** 32)
    rng = random.Random(seed + local_seed)

    choices = [wrong, correct]
    rng.shuffle(choices)

    label = choices.index(correct)

    q_text = (
        f"Math word problem: {question}\n"
        f"Choose the correct final numeric answer."
    )

    choice_texts = [f"The final answer is {c}" for c in choices]

    return q_text, choice_texts, label


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path)
        print(f"[cache] loaded rows: {len(rows)}")
        if len(rows) == 0:
            print("[cache] cached file is empty; rebuilding.")
            os.remove(path)
        else:
            return rows

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_mawps_example(ex, seed=cfg.seed)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"{q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(c_seqs, dim=0),
                "cmask": torch.stack(c_pads, dim=0),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check MAWPS parquet fields.")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, q_emb: torch.Tensor, c_emb: torch.Tensor, choice_mask: torch.Tensor) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat([q, c_emb, torch.abs(q - c_emb), q * c_emb], dim=-1)
        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)
        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        return self.head(q_emb, c_emb, choice_mask)


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)
        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device
        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        return self.head(q_emb, c_emb, choice_mask)


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(model: nn.Module, batch: Dict[str, torch.Tensor], device: torch.device):
    logits = model(batch)
    labels = batch["label"].to(device)
    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()
    return loss, acc


@torch.no_grad()
def evaluate(model: nn.Module, loader: Optional[DataLoader], device: torch.device, cfg: AblationConfig) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0
    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)
        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(model: nn.Module, ablation_name: str, cfg: AblationConfig, total_steps: int):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]
    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(model, ablation_name, cfg, total_steps)

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0
    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN MAWPS
# ============================================================

def run_mawps_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_mawps(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "test",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("MAWPS train dataset is empty. Check normalization/cache building.")

    if len(eval_ds) == 0:
        raise ValueError("MAWPS test dataset is empty. Check normalization/cache building.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = eval_loader

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())
        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")
    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: MAWPS")
    print("Train file:", cfg.hf_train_file)
    print("Test file:", cfg.hf_test_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_mawps_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: MAWPS
Train file: /home/user/twovolume/Nisha/Finetune/mawps/train-00000-of-00001.parquet
Test file: /home/user/twovolume/Nisha/Finetune/mawps/test-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: MAWPS ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MAWPS] train size: 1417
[MAWPS] test size: 355
[MAWPS] columns: ['question', 'answer']
[MAWPS] first example: {'question': "Alyssa 's dog had puppies . She gave 7 to her friends . She now has 5 puppies . How many puppies did she have to start with ?", 'answer': '7 + 5 = 12 #### 12'}
[cache] building MAWPS / train


cache:MAWPS:train: 100%|████████████████████████████████████████| 1417/1417 [01:34<00:00, 14.96it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=0.6932 train_acc=0.5025 eval_loss=0.6916 eval_acc=0.5268
[hlcm_frozen] saved best.pt
[hlcm_frozen][FINAL] best_eval_acc=0.5268 final_eval_acc=0.5268 final_test_acc=0.5268 time=5.68 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=0.6914 eval_acc=0.5127


hlcm_last2_unfrozen epoch 1/2: 100%|█| 709/709 [05:31<00:00,  2.14it/s, acc=0.5074, loss=0.6931, lr=


[hlcm_last2_unfrozen][epoch 1/2] train_loss=0.6931 train_acc=0.5074 eval_loss=0.6915 eval_acc=0.5465
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 709/709 [05:38<00:00,  2.09it/s, acc=0.5018, loss=0.6930, lr=


[hlcm_last2_unfrozen][epoch 2/2] train_loss=0.6930 train_acc=0.5018 eval_loss=0.6916 eval_acc=0.5070
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.5465 final_eval_acc=0.5465 final_test_acc=0.5465 time=12.99 min

[summary] saved runs/mawps_ablation/MAWPS/ablation_summary.csv
[summary] saved runs/mawps_ablation/MAWPS/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.5549 | test_acc=0.5549295774647888 | time=0.06 min
           hlcm_frozen | eval_acc=0.5268 | test_acc=0.5267605635481821 | time=5.68 min
   hlcm_last2_unfrozen | eval_acc=0.5465 | test_acc=0.546478873407337 | time=12.99 min

All done.
Outputs in: runs/mawps_ablation
Total wall time: 25.18 min


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("GSM8K",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    out_dir: str = "runs/gsm8k_ablation"
    cache_dir: str = "gsm8k_cached_features"

    rebuild_cache: bool = True

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, model_name: str, chunk_tok_len: int, seq_len: int, batch_size: int, device: torch.device):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# GSM8K DATA
# ============================================================

def load_gsm8k(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"GSM8K train file not found: {cfg.hf_train_file}")

    if not os.path.exists(cfg.hf_test_file):
        raise FileNotFoundError(f"GSM8K test file not found: {cfg.hf_test_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"]
    test_split = raw["test"]

    train_split = train_split.shuffle(seed=cfg.seed)

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        test_split = test_split.select(range(min(cfg.max_eval_examples, len(test_split))))

    print("[GSM8K] train size:", len(train_split))
    print("[GSM8K] test size:", len(test_split))
    print("[GSM8K] columns:", train_split.column_names)
    print("[GSM8K] first example:", train_split[0])

    return train_split, test_split, test_split


def extract_final_numeric_answer(answer_text: str) -> str:
    text = str(answer_text).replace(",", "")

    if "####" in text:
        final = text.split("####")[-1].strip()
    else:
        final = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final)
    if nums:
        return nums[-1]

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    raise ValueError(f"Could not extract numeric answer from: {answer_text}")


def make_wrong_numeric_answer(correct: str) -> str:
    try:
        if "." in correct:
            val = float(correct)
            wrong = val + 1.0 if val != -1.0 else val + 2.0
            return str(round(wrong, 4)).rstrip("0").rstrip(".")
        else:
            val = int(float(correct))
            wrong = val + 1 if val != -1 else val + 2
            return str(wrong)
    except Exception:
        return f"{correct}_wrong"


def normalize_gsm8k_example(ex: Dict[str, Any], seed: int = 42):
    question = ex.get("question", "")
    answer_text = ex.get("answer", "")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("GSM8K example has empty question.")

    correct = extract_final_numeric_answer(answer_text)
    wrong = make_wrong_numeric_answer(correct)

    key = str(question) + str(answer_text)
    local_seed = abs(hash(key)) % (2 ** 32)
    rng = random.Random(seed + local_seed)

    choices = [wrong, correct]
    rng.shuffle(choices)

    label = choices.index(correct)

    q_text = (
        f"Math word problem: {question}\n"
        f"Choose the correct final numeric answer."
    )

    choice_texts = [f"The final answer is {c}" for c in choices]

    return q_text, choice_texts, label


# ============================================================
# FEATURE CACHE
# ============================================================

def cache_file_path(cfg: AblationConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    required_keys = {
        "q",
        "qmask",
        "choices",
        "cmask",
        "choice_mask",
        "label",
        "num_choices",
    }

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path, map_location="cpu")
        print(f"[cache] loaded rows: {len(rows)}")

        if len(rows) > 0 and isinstance(rows[0], dict) and required_keys.issubset(rows[0].keys()):
            print("[cache] cache format is valid.")
            return rows

        print("[cache] old/incompatible cache detected; rebuilding.")
        os.remove(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label = normalize_gsm8k_example(ex, seed=cfg.seed)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            c_seqs = []
            c_pads = []

            for ct in choice_texts:
                qc = f"{q_text}\nAnswer Choice: {ct}"
                s, p = conceptizer.encode_text_fixed(qc)
                c_seqs.append(s)
                c_pads.append(p)

            K = len(c_seqs)

            rows.append({
                "q": q_seq.cpu().float(),
                "qmask": q_pad.cpu().bool(),
                "choices": torch.stack(c_seqs, dim=0).cpu().float(),
                "cmask": torch.stack(c_pads, dim=0).cpu().bool(),
                "choice_mask": torch.ones(K, dtype=torch.bool),
                "label": int(label),
                "num_choices": int(K),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    if len(rows) == 0:
        raise ValueError("Cache building produced 0 rows. Check GSM8K parquet fields.")

    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

        required_keys = {
            "q",
            "qmask",
            "choices",
            "cmask",
            "choice_mask",
            "label",
            "num_choices",
        }

        if len(rows) > 0 and not required_keys.issubset(rows[0].keys()):
            raise KeyError(
                f"Cached rows are missing keys. "
                f"Required={required_keys}, found={set(rows[0].keys())}. "
                f"Set rebuild_cache=True."
            )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    max_K = max(item["choices"].size(0) for item in batch)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, max_K, T, D, dtype=torch.float32)
    cmask = torch.ones(B, max_K, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, max_K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        K = item["choices"].size(0)
        choices[i, :K] = item["choices"]
        cmask[i, :K] = item["cmask"]
        choice_mask[i, :K] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: AblationConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# ABLATION MODEL
# ============================================================

def masked_mean_pool(x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class PairwiseMCQHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, q_emb: torch.Tensor, c_emb: torch.Tensor, choice_mask: torch.Tensor) -> torch.Tensor:
        B, K, D = c_emb.shape
        q = q_emb.unsqueeze(1).expand(B, K, D)

        pair = torch.cat([q, c_emb, torch.abs(q - c_emb), q * c_emb], dim=-1)
        logits = self.net(pair).squeeze(-1)

        min_val = torch.finfo(logits.dtype).min
        logits = logits.masked_fill(~choice_mask.to(logits.device), min_val)
        return logits


class DebertaOnlyMCQModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = PairwiseMCQHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = masked_mean_pool(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = masked_mean_pool(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, D)

        return self.head(q_emb, c_emb, choice_mask)


class HLCMMCQAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.hlcm = hlcm
        self.head = PairwiseMCQHead(dim=model_dim, dropout=dropout)
        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(self, x: torch.Tensor, pad_mask: torch.Tensor) -> torch.Tensor:
        device = next(self.parameters()).device
        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        q = batch["q"].to(device)
        qmask = batch["qmask"].to(device)
        choices = batch["choices"].to(device)
        cmask = batch["cmask"].to(device)
        choice_mask = batch["choice_mask"].to(device)

        B, K, T, D = choices.shape

        q_emb = self.encode_hlcm(q, qmask)

        flat_choices = choices.reshape(B * K, T, D)
        flat_cmask = cmask.reshape(B * K, T)

        c_emb = self.encode_hlcm(flat_choices, flat_cmask)
        c_emb = c_emb.reshape(B, K, -1)

        q_emb = F.normalize(q_emb, dim=-1)
        c_emb = F.normalize(c_emb, dim=-1)

        return self.head(q_emb, c_emb, choice_mask)


def freeze_hlcm_backbone(model: HLCMMCQAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(model: HLCMMCQAblationModel, n_last: int):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL
# ============================================================

def compute_loss_acc(model: nn.Module, batch: Dict[str, torch.Tensor], device: torch.device):
    logits = model(batch)
    labels = batch["label"].to(device)
    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=1) == labels).float().mean()
    return loss, acc


@torch.no_grad()
def evaluate(model: nn.Module, loader: Optional[DataLoader], device: torch.device, cfg: AblationConfig) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0
    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                loss, acc = compute_loss_acc(model, batch, device)
        else:
            loss, acc = compute_loss_acc(model, batch, device)

        bs = batch["label"].size(0)
        total_loss += float(loss.item()) * bs
        total_acc += float(acc.item()) * bs
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "acc": total_acc / max(1, total_n),
    }


def make_optimizer_and_scheduler(model: nn.Module, ablation_name: str, cfg: AblationConfig, total_steps: int):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]
    else:
        param_groups = [
            {"params": [p for p in model.parameters() if p.requires_grad], "lr": cfg.head_lr},
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(model, ablation_name, cfg, total_steps)

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)
    print(f"[BASE] eval_loss={base_eval['loss']:.4f} eval_acc={base_eval['acc']:.4f}")

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base_eval["loss"],
        "eval_acc": base_eval["acc"],
    })

    best_acc = base_eval["acc"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = compute_loss_acc(model, batch, device)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = compute_loss_acc(model, batch, device)
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "global_step": global_step,
            "lr": opt.param_groups[0]["lr"],
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": eval_result["loss"],
            "eval_acc": eval_result["acc"],
        })

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_acc={eval_result['acc']:.4f}"
        )

        if eval_result["acc"] > best_acc:
            best_acc = eval_result["acc"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_acc": best_acc,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0
    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_acc": best_acc,
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_acc": None if final_test is None else final_test["acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"best_eval_acc={best_acc:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        + (f"final_test_acc={final_test['acc']:.4f} " if final_test is not None else "")
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN GSM8K
# ============================================================

def run_gsm8k_ablation(dataset_name: str, cfg: AblationConfig, device: torch.device):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_gsm8k(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "test",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)
    test_ds = CachedMCQDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("GSM8K train dataset is empty. Check normalization/cache building.")

    if len(eval_ds) == 0:
        raise ValueError("GSM8K test dataset is empty. Check normalization/cache building.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = eval_loader

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())
        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")
    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")
    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_acc={row['final_eval_acc']:.4f} | "
            f"test_acc={row['final_test_acc'] if row['final_test_acc'] is not None else 'NA'} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: GSM8K")
    print("Train file:", cfg.hf_train_file)
    print("Test file:", cfg.hf_test_file)
    print("Ablations:", cfg.ablations_to_run)
    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_gsm8k_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: GSM8K
Train file: /home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet
Test file: /home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: GSM8K ====================
[cache] rebuild_cache=True. Removing cache directory: gsm8k_cached_features


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[GSM8K] train size: 7473
[GSM8K] test size: 1319
[GSM8K] columns: ['question', 'answer']
[GSM8K] first example: {'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16'}
[cache] building GSM8K / train


cache:GSM8K:train: 100%|████████████████████████████████████████| 7473/7473 [10:17<00:00, 12.10it/s]


...

[hlcm_frozen][epoch 2/2] train_loss=0.6926 train_acc=0.5132 eval_loss=0.6921 eval_acc=0.6937
[hlcm_frozen] saved best.pt
[hlcm_frozen][FINAL] best_eval_acc=0.6937 final_eval_acc=0.6937 final_test_acc=0.6937 time=22.56 min

[build] ablation=hlcm_last2_unfrozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_last2_unfrozen ==========
[params] total=2,486,824,962 trainable=469,876,738
[BASE] eval_loss=0.6914 eval_acc=0.5208


hlcm_last2_unfrozen epoch 1/2: 100%|█| 3737/3737 [29:00<00:00,  2.15it/s, acc=0.5109, loss=0.6930, l


[hlcm_last2_unfrozen][epoch 1/2] train_loss=0.6930 train_acc=0.5109 eval_loss=0.6921 eval_acc=0.6960
[hlcm_last2_unfrozen] saved best.pt


hlcm_last2_unfrozen epoch 2/2: 100%|█| 3737/3737 [29:39<00:00,  2.10it/s, acc=0.5088, loss=0.6928, l


[hlcm_last2_unfrozen][epoch 2/2] train_loss=0.6928 train_acc=0.5088 eval_loss=0.6922 eval_acc=0.7036
[hlcm_last2_unfrozen] saved best.pt
[hlcm_last2_unfrozen][FINAL] best_eval_acc=0.7036 final_eval_acc=0.7036 final_test_acc=0.7036 time=62.00 min

[summary] saved runs/gsm8k_ablation/GSM8K/ablation_summary.csv
[summary] saved runs/gsm8k_ablation/GSM8K/ablation_summary.json

========== FINAL ABLATION TABLE ==========
          deberta_only | eval_acc=0.7460 | test_acc=0.7460197119255514 | time=0.27 min
           hlcm_frozen | eval_acc=0.6937 | test_acc=0.6937073541012924 | time=22.56 min
   hlcm_last2_unfrozen | eval_acc=0.7036 | test_acc=0.703563305579685 | time=62.00 min

All done.
Outputs in: runs/gsm8k_ablation
Total wall time: 105.03 min


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
import shutil
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class AblationConfig:
    datasets_to_run: Tuple[str, ...] = ("GSM8K",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    out_dir: str = "runs/gsm8k_regression_ablation"
    cache_dir: str = "gsm8k_regression_cached_features"

    # IMPORTANT:
    # Set True once to delete old incompatible cache files.
    # After successful cache rebuild, you can set it back to False.
    rebuild_cache: bool = True

    ablations_to_run: Tuple[str, ...] = (
        "hlcm_frozen",
        "hlcm_last2_unfrozen",
    )

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    epochs: int = 2
    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    n_last_blocks_unfrozen: int = 2
    head_dropout: float = 0.10

    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    if path:
        os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_checkpoint(payload: dict, path: str):
    ensure_dir(os.path.dirname(path))
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))
    file_exists = os.path.isfile(path)

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    ensure_dir(os.path.dirname(path))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {
            "alloc_mb": 0.0,
            "reserved_mb": 0.0,
            "max_alloc_mb": 0.0,
        }

    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()

    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }


def set_requires_grad(module: nn.Module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )

        inputs = {
            k: v.to(self.device)
            for k, v in inputs.items()
        }

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        out = []

        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )

            if len(out) >= self.seq_len:
                break

        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(
            self.seq_len,
            self.hidden_size,
            dtype=torch.float32,
        )

        pad = torch.ones(
            self.seq_len,
            dtype=torch.bool,
        )

        if not chunk_texts:
            return seq, pad

        vecs = []

        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(
                self._embed_chunk_texts(
                    chunk_texts[i:i + self.batch_size]
                )
            )

        vecs = torch.cat(vecs, dim=0)

        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# GSM8K DATA
# ============================================================

def load_gsm8k(cfg: AblationConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(
            f"GSM8K train file not found: {cfg.hf_train_file}"
        )

    if not os.path.exists(cfg.hf_test_file):
        raise FileNotFoundError(
            f"GSM8K test file not found: {cfg.hf_test_file}"
        )

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"].shuffle(seed=cfg.seed)
    test_split = raw["test"]

    if cfg.max_train_examples is not None:
        train_split = train_split.select(
            range(min(cfg.max_train_examples, len(train_split)))
        )

    if cfg.max_eval_examples is not None:
        test_split = test_split.select(
            range(min(cfg.max_eval_examples, len(test_split)))
        )

    print("[GSM8K] train size:", len(train_split))
    print("[GSM8K] test size:", len(test_split))
    print("[GSM8K] columns:", train_split.column_names)
    print("[GSM8K] first example:", train_split[0])

    return train_split, test_split, test_split


def extract_final_numeric_answer(answer_text: str) -> float:
    text = str(answer_text).replace(",", "")

    if "####" in text:
        final_part = text.split("####")[-1].strip()
    else:
        final_part = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final_part)
    if nums:
        return float(nums[-1])

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return float(nums[-1])

    raise ValueError(
        f"Could not extract numeric answer from: {answer_text}"
    )


def answer_to_log_value(y: float) -> float:
    sign = 1.0 if y >= 0 else -1.0
    return sign * math.log1p(abs(y))


def log_value_to_answer(z: torch.Tensor) -> torch.Tensor:
    sign = torch.sign(z)
    sign = torch.where(sign == 0, torch.ones_like(sign), sign)
    return sign * torch.expm1(torch.abs(z))


def normalize_gsm8k_example(ex: Dict[str, Any]):
    question = ex.get("question", "")
    answer_text = ex.get("answer", "")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("GSM8K example has empty question.")

    y_raw = extract_final_numeric_answer(answer_text)
    y_log = answer_to_log_value(y_raw)

    input_text = (
        f"Math word problem: {question}\n"
        f"Solve the problem step by step and predict the final numeric answer."
    )

    return input_text, float(y_raw), float(y_log)

# ============================================================
# FEATURE CACHE FOR REGRESSION
# ============================================================

def cache_file_path(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    suffix = ""

    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_regression_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cached_split(
    cfg: AblationConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    required_keys = {"x", "mask", "y_raw", "y_log"}

    if cfg.rebuild_cache and os.path.exists(path):
        print(f"[cache] deleting old cache: {path}")
        os.remove(path)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        rows = torch.load(path, map_location="cpu")
        print(f"[cache] loaded rows: {len(rows)}")

        if len(rows) > 0 and isinstance(rows[0], dict):
            if required_keys.issubset(rows[0].keys()):
                print("[cache] cache format is valid.")
                return rows

        print("[cache] old/incompatible cache detected; rebuilding.")
        os.remove(path)

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            text, y_raw, y_log = normalize_gsm8k_example(ex)
            seq, pad = conceptizer.encode_text_fixed(text)

            rows.append(
                {
                    "x": seq.cpu().float(),
                    "mask": pad.cpu().bool(),
                    "y_raw": float(y_raw),
                    "y_log": float(y_log),
                }
            )

        except Exception as e:
            skipped += 1

            if skipped <= 10:
                print(
                    f"[warn] skipped one example: "
                    f"{type(e).__name__}: {e}"
                )

    torch.save(rows, path)

    print(
        f"[cache] saved {path} "
        f"({len(rows)} examples, skipped={skipped})"
    )

    if len(rows) == 0:
        raise ValueError(
            "Cache building produced 0 rows. Check GSM8K parquet fields."
        )

    return rows


class CachedRegressionDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

        required_keys = {"x", "mask", "y_raw", "y_log"}

        if len(rows) > 0 and not required_keys.issubset(rows[0].keys()):
            raise KeyError(
                f"Cached rows are missing keys. "
                f"Required={required_keys}, "
                f"found={set(rows[0].keys())}. "
                f"Delete the cache or set rebuild_cache=True."
            )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "x": r["x"],
            "mask": r["mask"],
            "y_raw": torch.tensor(r["y_raw"], dtype=torch.float32),
            "y_log": torch.tensor(r["y_log"], dtype=torch.float32),
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    x = torch.stack([item["x"] for item in batch], dim=0)
    mask = torch.stack([item["mask"] for item in batch], dim=0)
    y_raw = torch.stack([item["y_raw"] for item in batch], dim=0)
    y_log = torch.stack([item["y_log"] for item in batch], dim=0)
    idxs = torch.tensor([item["idx"] for item in batch], dtype=torch.long)

    return {
        "x": x,
        "mask": mask,
        "y_raw": y_raw,
        "y_log": y_log,
        "idx": idxs,
    }


# ============================================================
# H-LCM LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: AblationConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(
    cfg: AblationConfig,
    device: torch.device,
) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {cfg.ckpt_path}"
        )

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded H-LCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# REGRESSION MODELS
# ============================================================

def masked_mean_pool(
    x: torch.Tensor,
    pad_mask: torch.Tensor,
) -> torch.Tensor:
    valid = (~pad_mask).float().unsqueeze(-1)
    denom = valid.sum(dim=1).clamp_min(1.0)
    return (x * valid).sum(dim=1) / denom


class RegressionHead(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, emb: torch.Tensor) -> torch.Tensor:
        return self.net(emb).squeeze(-1)


class DebertaOnlyRegressionModel(nn.Module):
    def __init__(self, in_dim: int = 768, dropout: float = 0.1):
        super().__init__()
        self.head = RegressionHead(dim=in_dim, dropout=dropout)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        device = next(self.parameters()).device

        x = batch["x"].to(device)
        mask = batch["mask"].to(device)

        emb = masked_mean_pool(x, mask)
        pred_log = self.head(emb)

        return pred_log


class HLCMRegressionAblationModel(nn.Module):
    def __init__(
        self,
        hlcm: HyperbolicLCM,
        model_dim: int,
        dropout: float = 0.1,
        mu: Optional[torch.Tensor] = None,
        sigma: Optional[torch.Tensor] = None,
    ):
        super().__init__()

        self.hlcm = hlcm
        self.head = RegressionHead(dim=model_dim, dropout=dropout)

        self.mu = mu
        self.sigma = sigma

    def encode_hlcm(
        self,
        x: torch.Tensor,
        pad_mask: torch.Tensor,
    ) -> torch.Tensor:
        device = next(self.parameters()).device

        x = x.to(device)
        pad_mask = pad_mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape

        out = torch.empty(
            B,
            D,
            device=h_tan.device,
            dtype=h_tan.dtype,
        )

        for i in range(B):
            valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return out

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = batch["x"]
        mask = batch["mask"]

        emb = self.encode_hlcm(x, mask)
        emb = F.normalize(emb, dim=-1)

        pred_log = self.head(emb)

        return pred_log


def freeze_hlcm_backbone(model: HLCMRegressionAblationModel):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)


def unfreeze_last_hlcm_blocks(
    model: HLCMRegressionAblationModel,
    n_last: int,
):
    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError(
            "HyperbolicLCM is expected to have attribute 'layers'."
        )

    layers = list(model.hlcm.layers)
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)


# ============================================================
# TRAIN / EVAL FOR REGRESSION
# ============================================================

def compute_loss_metrics(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    device: torch.device,
):
    pred_log = model(batch)

    y_log = batch["y_log"].to(device)
    y_raw = batch["y_raw"].to(device)

    loss = F.smooth_l1_loss(pred_log, y_log)

    pred_raw = log_value_to_answer(pred_log.float())

    mae = torch.mean(torch.abs(pred_raw - y_raw)).item()
    rmse = torch.sqrt(torch.mean((pred_raw - y_raw) ** 2)).item()
    rounded_acc = (
        torch.round(pred_raw) == torch.round(y_raw)
    ).float().mean().item()

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "rounded_acc": rounded_acc,
    }

    return loss, metrics


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: Optional[DataLoader],
    device: torch.device,
    cfg: AblationConfig,
) -> Dict[str, float]:
    if loader is None:
        return {
            "loss": 0.0,
            "mae": 0.0,
            "rmse": 0.0,
            "rounded_acc": 0.0,
        }

    model.eval()

    total_loss = 0.0
    total_abs = 0.0
    total_sq = 0.0
    total_exact = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                pred_log = model(batch)
                y_log = batch["y_log"].to(device)
                y_raw = batch["y_raw"].to(device)
                loss = F.smooth_l1_loss(pred_log, y_log)
        else:
            pred_log = model(batch)
            y_log = batch["y_log"].to(device)
            y_raw = batch["y_raw"].to(device)
            loss = F.smooth_l1_loss(pred_log, y_log)

        pred_raw = log_value_to_answer(pred_log.float())

        bs = y_raw.size(0)

        total_loss += float(loss.item()) * bs
        total_abs += torch.sum(torch.abs(pred_raw - y_raw)).item()
        total_sq += torch.sum((pred_raw - y_raw) ** 2).item()
        total_exact += torch.sum(
            torch.round(pred_raw) == torch.round(y_raw)
        ).item()
        total_n += bs

    mae = total_abs / max(1, total_n)
    rmse = math.sqrt(total_sq / max(1, total_n))
    rounded_acc = total_exact / max(1, total_n)

    return {
        "loss": total_loss / max(1, total_n),
        "mae": mae,
        "rmse": rmse,
        "rounded_acc": rounded_acc,
    }
def make_optimizer_and_scheduler(
    model: nn.Module,
    ablation_name: str,
    cfg: AblationConfig,
    total_steps: int,
):
    if ablation_name == "hlcm_last2_unfrozen":
        head_params = [p for p in model.head.parameters() if p.requires_grad]
        backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

        param_groups = [
            {"params": head_params, "lr": cfg.head_lr},
            {"params": backbone_params, "lr": cfg.backbone_lr},
        ]
    else:
        param_groups = [
            {
                "params": [p for p in model.parameters() if p.requires_grad],
                "lr": cfg.head_lr,
            }
        ]

    trainable_count = sum(
        p.numel()
        for group in param_groups
        for p in group["params"]
        if p.requires_grad
    )

    if trainable_count == 0:
        raise ValueError(f"No trainable parameters for ablation: {ablation_name}")

    opt = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


def train_one_ablation(
    model: nn.Module,
    ablation_name: str,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    test_loader: Optional[DataLoader],
    out_dir: str,
    cfg: AblationConfig,
    device: torch.device,
):
    ensure_dir(out_dir)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n========== Ablation: {ablation_name} ==========")
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_and_scheduler(
        model=model,
        ablation_name=ablation_name,
        cfg=cfg,
        total_steps=total_steps,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "train_log.csv")
    eval_csv = os.path.join(out_dir, "eval_log.csv")

    base_eval = evaluate(model, eval_loader, device, cfg)

    print(
        f"[BASE] loss={base_eval['loss']:.4f} "
        f"mae={base_eval['mae']:.4f} "
        f"rmse={base_eval['rmse']:.4f} "
        f"rounded_acc={base_eval['rounded_acc']:.4f}"
    )

    append_dict_to_csv(
        eval_csv,
        {
            "phase": "base",
            "epoch": 0,
            "train_loss": "",
            "train_mae": "",
            "train_rmse": "",
            "train_rounded_acc": "",
            "eval_loss": base_eval["loss"],
            "eval_mae": base_eval["mae"],
            "eval_rmse": base_eval["rmse"],
            "eval_rounded_acc": base_eval["rounded_acc"],
        },
    )

    best_mae = base_eval["mae"]
    best_state = clone_state_dict_to_cpu(model)

    t0 = time.time()
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_abs_sum = 0.0
        epoch_sq_sum = 0.0
        epoch_exact_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"{ablation_name} epoch {epoch}/{cfg.epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    pred_log = model(batch)
                    y_log = batch["y_log"].to(device)
                    y_raw = batch["y_raw"].to(device)
                    loss = F.smooth_l1_loss(pred_log, y_log)
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                pred_log = model(batch)
                y_log = batch["y_log"].to(device)
                y_raw = batch["y_raw"].to(device)
                loss = F.smooth_l1_loss(pred_log, y_log)
                loss_for_backward = loss / cfg.grad_accum_steps

            pred_raw = log_value_to_answer(pred_log.float())

            bs = y_raw.size(0)

            epoch_loss_sum += float(loss.item()) * bs
            epoch_abs_sum += torch.sum(torch.abs(pred_raw - y_raw)).item()
            epoch_sq_sum += torch.sum((pred_raw - y_raw) ** 2).item()
            epoch_exact_sum += torch.sum(torch.round(pred_raw) == torch.round(y_raw)).item()
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (
                batch_idx == len(train_loader)
            )

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            train_loss_running = epoch_loss_sum / max(1, epoch_count)
            train_mae_running = epoch_abs_sum / max(1, epoch_count)
            train_rmse_running = math.sqrt(epoch_sq_sum / max(1, epoch_count))
            train_acc_running = epoch_exact_sum / max(1, epoch_count)

            pbar.set_postfix(
                loss=f"{train_loss_running:.4f}",
                mae=f"{train_mae_running:.4f}",
                rmse=f"{train_rmse_running:.4f}",
                acc=f"{train_acc_running:.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_mae = epoch_abs_sum / max(1, epoch_count)
        train_rmse = math.sqrt(epoch_sq_sum / max(1, epoch_count))
        train_rounded_acc = epoch_exact_sum / max(1, epoch_count)

        eval_result = evaluate(model, eval_loader, device, cfg)
        cuda_cleanup()

        append_dict_to_csv(
            train_csv,
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_mae": train_mae,
                "train_rmse": train_rmse,
                "train_rounded_acc": train_rounded_acc,
                "global_step": global_step,
                "lr": opt.param_groups[0]["lr"],
            },
        )

        append_dict_to_csv(
            eval_csv,
            {
                "phase": "eval",
                "epoch": epoch,
                "train_loss": train_loss,
                "train_mae": train_mae,
                "train_rmse": train_rmse,
                "train_rounded_acc": train_rounded_acc,
                "eval_loss": eval_result["loss"],
                "eval_mae": eval_result["mae"],
                "eval_rmse": eval_result["rmse"],
                "eval_rounded_acc": eval_result["rounded_acc"],
            },
        )

        print(
            f"[{ablation_name}][epoch {epoch}/{cfg.epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_mae={train_mae:.4f} "
            f"train_rmse={train_rmse:.4f} "
            f"train_rounded_acc={train_rounded_acc:.4f} "
            f"eval_loss={eval_result['loss']:.4f} "
            f"eval_mae={eval_result['mae']:.4f} "
            f"eval_rmse={eval_result['rmse']:.4f} "
            f"eval_rounded_acc={eval_result['rounded_acc']:.4f}"
        )

        if eval_result["mae"] < best_mae:
            best_mae = eval_result["mae"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "ablation": ablation_name,
                    "epoch": epoch,
                    "best_eval_mae": best_mae,
                    "global_step": global_step,
                },
                os.path.join(out_dir, "best.pt"),
            )

            print(f"[{ablation_name}] saved best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "ablation": ablation_name,
                "epoch": epoch,
                "global_step": global_step,
            },
            os.path.join(out_dir, "last.pt"),
        )

    total_minutes = (time.time() - t0) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_eval = evaluate(model, eval_loader, device, cfg)
    final_test = evaluate(model, test_loader, device, cfg) if test_loader is not None else None

    mem = gpu_mem_mb(device)

    summary = {
        "ablation": ablation_name,
        "best_eval_mae": best_mae,
        "final_eval_loss": final_eval["loss"],
        "final_eval_mae": final_eval["mae"],
        "final_eval_rmse": final_eval["rmse"],
        "final_eval_rounded_acc": final_eval["rounded_acc"],
        "final_test_loss": None if final_test is None else final_test["loss"],
        "final_test_mae": None if final_test is None else final_test["mae"],
        "final_test_rmse": None if final_test is None else final_test["rmse"],
        "final_test_rounded_acc": None if final_test is None else final_test["rounded_acc"],
        "total_minutes": total_minutes,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "epochs": cfg.epochs,
        "head_lr": cfg.head_lr,
        "backbone_lr": cfg.backbone_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "max_train_examples": cfg.max_train_examples,
        "max_eval_examples": cfg.max_eval_examples,
    }

    write_single_row_csv(os.path.join(out_dir, "summary.csv"), summary)

    with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"[{ablation_name}][FINAL] "
        f"eval_mae={final_eval['mae']:.4f} "
        f"eval_rmse={final_eval['rmse']:.4f} "
        f"eval_rounded_acc={final_eval['rounded_acc']:.4f} "
        + (
            f"test_mae={final_test['mae']:.4f} "
            f"test_rmse={final_test['rmse']:.4f} "
            f"test_rounded_acc={final_test['rounded_acc']:.4f} "
            if final_test is not None
            else ""
        )
        + f"time={total_minutes:.2f} min"
    )

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return summary


# ============================================================
# BUILD ABLATION MODEL
# ============================================================

def build_ablation_model(
    ablation_name: str,
    cfg: AblationConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> nn.Module:
    if ablation_name in {"hlcm_frozen", "hlcm_last2_unfrozen"}:
        hlcm = load_pretrained_hlcm(cfg, device)

        model = HLCMMCQAblationModel(
            hlcm=hlcm,
            model_dim=cfg.model_dim,
            dropout=cfg.head_dropout,
            mu=mu,
            sigma=sigma,
        ).to(device)

        if ablation_name == "hlcm_frozen":
            freeze_hlcm_backbone(model)

        elif ablation_name == "hlcm_last2_unfrozen":
            unfreeze_last_hlcm_blocks(
                model,
                n_last=cfg.n_last_blocks_unfrozen,
            )

        return model

    raise ValueError(f"Unknown ablation name: {ablation_name}")


# ============================================================
# RUN GSM8K
# ============================================================

def run_gsm8k_ablation(
    dataset_name: str,
    cfg: AblationConfig,
    device: torch.device,
):
    print(f"\n==================== Dataset: {dataset_name} ====================")

    if cfg.rebuild_cache:
        print(f"[cache] rebuild_cache=True. Removing cache directory: {cfg.cache_dir}")
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)

    dataset_out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(dataset_out_dir)
    ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=torch.device(cfg.conceptizer_device),
    )

    train_hf, eval_hf, test_hf = load_gsm8k(cfg)

    train_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "train",
        train_hf,
        conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg,
        dataset_name,
        "test",
        eval_hf,
        conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedRegressionDataset(train_rows)
    eval_ds = CachedRegressionDataset(eval_rows)
    test_ds = CachedRegressionDataset(eval_rows)

    if len(train_ds) == 0:
        raise ValueError("GSM8K train dataset is empty.")

    if len(eval_ds) == 0:
        raise ValueError("GSM8K test dataset is empty.")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    test_loader = eval_loader

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    all_summaries = []

    for ablation_name in cfg.ablations_to_run:
        ablation_out_dir = os.path.join(dataset_out_dir, ablation_name)
        ensure_dir(ablation_out_dir)

        print(f"\n[build] ablation={ablation_name}")

        model = build_ablation_model(
            ablation_name=ablation_name,
            cfg=cfg,
            device=device,
            mu=mu,
            sigma=sigma,
        )

        summary = train_one_ablation(
            model=model,
            ablation_name=ablation_name,
            train_loader=train_loader,
            eval_loader=eval_loader,
            test_loader=test_loader,
            out_dir=ablation_out_dir,
            cfg=cfg,
            device=device,
        )

        summary["dataset"] = dataset_name
        all_summaries.append(summary)

        del model
        cuda_cleanup()

    combined_csv = os.path.join(dataset_out_dir, "ablation_summary.csv")

    if all_summaries:
        fieldnames = list(all_summaries[0].keys())

        with open(combined_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_summaries)

    combined_json = os.path.join(dataset_out_dir, "ablation_summary.json")

    with open(combined_json, "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2)

    print(f"\n[summary] saved {combined_csv}")
    print(f"[summary] saved {combined_json}")

    print("\n========== FINAL ABLATION TABLE ==========")

    for row in all_summaries:
        print(
            f"{row['ablation']:>22s} | "
            f"eval_mae={row['final_eval_mae']:.4f} | "
            f"eval_rmse={row['final_eval_rmse']:.4f} | "
            f"eval_rounded_acc={row['final_eval_rounded_acc']:.4f} | "
            f"test_mae={row['final_test_mae']:.4f} | "
            f"test_rounded_acc={row['final_test_rounded_acc']:.4f} | "
            f"time={row['total_minutes']:.2f} min"
        )


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = AblationConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Dataset: GSM8K")
    print("Train file:", cfg.hf_train_file)
    print("Test file:", cfg.hf_test_file)
    print("Ablations:", cfg.ablations_to_run)

    print(
        f"epochs={cfg.epochs}, "
        f"bs_train={cfg.train_batch_size}, "
        f"bs_eval={cfg.eval_batch_size}, "
        f"grad_accum={cfg.grad_accum_steps}"
    )

    print(
        f"head_lr={cfg.head_lr}, "
        f"backbone_lr={cfg.backbone_lr}, "
        f"last_unfrozen_blocks={cfg.n_last_blocks_unfrozen}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        run_gsm8k_ablation(ds_name, cfg, device)

    total_minutes = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_minutes:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Dataset: GSM8K
Train file: /home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet
Test file: /home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet
Ablations: ('deberta_only', 'hlcm_frozen', 'hlcm_last2_unfrozen')
epochs=2, bs_train=2, bs_eval=4, grad_accum=8
head_lr=5e-05, backbone_lr=1e-05, last_unfrozen_blocks=2

==================== Dataset: GSM8K ====================
[cache] rebuild_cache=True. Removing cache directory: gsm8k_regression_cached_features


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[GSM8K] train size: 7473
[GSM8K] test size: 1319
[GSM8K] columns: ['question', 'answer']
[GSM8K] first example: {'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16'}
[cache] building GSM8K / train


cache:GSM8K:train: 100%|████████████████████████████████████████| 7473/7473 [03:26<00:00, 36.11it/s]


...

deberta_only epoch 1/2: 100%|█| 3737/3737 [00:07<00:00, 490.98it/s, acc=0.0048, loss=1.2931, lr=2.62


[deberta_only][epoch 1/2] train_loss=1.2931 train_mae=53926.9862 train_rmse=2582275.4163 train_rounded_acc=0.0048 eval_loss=1.1377 eval_mae=6812.6039 eval_rmse=92049.6274 eval_rounded_acc=0.0015
[deberta_only] saved best.pt


deberta_only epoch 2/2: 100%|█| 3737/3737 [00:06<00:00, 537.27it/s, acc=0.0047, loss=1.1278, lr=0.00


[deberta_only][epoch 2/2] train_loss=1.1278 train_mae=53923.7172 train_rmse=2582274.5441 train_rounded_acc=0.0047 eval_loss=1.1259 eval_mae=6812.0656 eval_rmse=92049.6252 eval_rounded_acc=0.0068
[deberta_only] saved best.pt
[deberta_only][FINAL] eval_mae=6812.0656 eval_rmse=92049.6252 eval_rounded_acc=0.0068 test_mae=6812.0656 test_rmse=92049.6252 test_rounded_acc=0.0068 time=0.26 min

[build] ablation=hlcm_frozen
[load] loaded H-LCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0

========== Ablation: hlcm_frozen ==========
[params] total=2,436,501,506 trainable=16,793,601


KeyboardInterrupt: 

In [1]:
# GSM8K Hyperbolic Reasoning Training
# Question -> H-LCM -> final numeric answer
# Training also aligns question representation with solution/reasoning representation.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import re
import gc
import csv
import json
import math
import time
import random
import shutil
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    dataset_name: str = "GSM8K"

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/train-00000-of-00001.parquet"
    hf_test_file: str = "/home/user/twovolume/Nisha/Finetune/gsm8k/test-00000-of-00001.parquet"

    out_dir: str = "runs/gsm8k_hlcm_math_reasoning"
    cache_dir: str = "gsm8k_hlcm_math_reasoning_cache"
    rebuild_cache: bool = False

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 8
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    epochs: int = 3
    num_workers: int = 0

    head_lr: float = 5e-5
    backbone_lr: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # Reasoning objective
    answer_loss_weight: float = 1.0
    reasoning_loss_weight: float = 0.3
    contrastive_temperature: float = 0.07

    # Fine-tuning mode
    freeze_hlcm: bool = False
    n_last_blocks_unfrozen: int = 2

    head_dropout: float = 0.10
    max_train_examples: Optional[int] = None
    max_eval_examples: Optional[int] = None

    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# UTILS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def pick_device(idx=0):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = max(0, min(idx, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device):
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def set_requires_grad(module, flag):
    for p in module.parameters():
        p.requires_grad = flag


def clone_state_dict_to_cpu(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def load_normalizer(path, device):
    if not path or not os.path.exists(path):
        print("[normalizer] not found; using raw DeBERTa embeddings.")
        return None, None

    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    print(f"[normalizer] loaded from {path}")
    return mu, sigma


def extract_final_numeric_answer(answer_text: str) -> float:
    text = str(answer_text).replace(",", "")

    if "####" in text:
        final = text.split("####")[-1].strip()
    else:
        final = text.strip()

    nums = re.findall(r"-?\d+(?:\.\d+)?", final)
    if nums:
        return float(nums[-1])

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return float(nums[-1])

    raise ValueError(f"Could not extract answer from: {answer_text}")


def answer_to_log_value(y: float) -> float:
    sign = 1.0 if y >= 0 else -1.0
    return sign * math.log1p(abs(y))


def log_value_to_answer(z: torch.Tensor) -> torch.Tensor:
    sign = torch.sign(z)
    sign = torch.where(sign == 0, torch.ones_like(sign), sign)
    return sign * torch.expm1(torch.abs(z))


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(self, cfg: Config):
        self.device = torch.device(cfg.conceptizer_device)
        self.chunk_tok_len = cfg.chunk_tok_len
        self.seq_len = cfg.seq_len
        self.batch_size = cfg.encoder_batch_size

        self.tok = AutoTokenizer.from_pretrained(cfg.encoder_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(cfg.encoder_name).to(self.device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    def _chunks(self, text):
        ids = self.tok(str(text), add_special_tokens=False)["input_ids"]
        chunks = []

        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break

        return chunks

    @torch.inference_mode()
    def _embed(self, texts):
        if len(texts) == 0:
            return torch.empty(0, self.hidden_size)

        inputs = self.tok(
            texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    @torch.inference_mode()
    def encode_fixed(self, text):
        chunks = self._chunks(text)

        seq = torch.zeros(self.seq_len, self.hidden_size)
        mask = torch.ones(self.seq_len, dtype=torch.bool)

        if len(chunks) == 0:
            return seq, mask

        vecs = []
        for i in range(0, len(chunks), self.batch_size):
            vecs.append(self._embed(chunks[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        n = min(vecs.size(0), self.seq_len)

        seq[:n] = vecs[:n]
        mask[:n] = False

        return seq, mask


# ============================================================
# DATA
# ============================================================

def load_gsm8k(cfg: Config):
    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "test": cfg.hf_test_file,
        },
    )

    train_split = raw["train"].shuffle(seed=cfg.seed)
    test_split = raw["test"]

    if cfg.max_train_examples is not None:
        train_split = train_split.select(range(min(cfg.max_train_examples, len(train_split))))

    if cfg.max_eval_examples is not None:
        test_split = test_split.select(range(min(cfg.max_eval_examples, len(test_split))))

    print("[GSM8K] train:", len(train_split))
    print("[GSM8K] test:", len(test_split))
    print("[GSM8K] columns:", train_split.column_names)
    print("[GSM8K] first example:", train_split[0])

    return train_split, test_split


def cache_path(cfg: Config, split_name: str):
    suffix = ""
    if cfg.max_train_examples is not None or cfg.max_eval_examples is not None:
        suffix = f"_train{cfg.max_train_examples}_eval{cfg.max_eval_examples}"

    return os.path.join(
        cfg.cache_dir,
        f"gsm8k_reasoning_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}{suffix}.pt",
    )


def build_or_load_cache(cfg: Config, split_name: str, hf_split, conceptizer):
    ensure_dir(cfg.cache_dir)
    path = cache_path(cfg, split_name)

    if cfg.rebuild_cache and os.path.exists(path):
        os.remove(path)

    if os.path.exists(path):
        rows = torch.load(path)
        print(f"[cache] loaded {path}: {len(rows)} rows")
        if len(rows) > 0:
            return rows
        os.remove(path)

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            question = str(ex["question"])
            solution = str(ex["answer"])
            y_raw = extract_final_numeric_answer(solution)
            y_log = answer_to_log_value(y_raw)

            q_text = (
                f"Math word problem:\n{question}\n"
                f"Reason through the problem and predict the final numeric answer."
            )

            reasoning_text = (
                f"Problem:\n{question}\n"
                f"Reasoning solution:\n{solution}"
            )

            q_seq, q_mask = conceptizer.encode_fixed(q_text)
            r_seq, r_mask = conceptizer.encode_fixed(reasoning_text)

            rows.append({
                "q": q_seq,
                "qmask": q_mask,
                "reason": r_seq,
                "reason_mask": r_mask,
                "y_raw": float(y_raw),
                "y_log": float(y_log),
            })

        except Exception as e:
            skipped += 1
            if skipped <= 5:
                print("[skip]", type(e).__name__, e)

    torch.save(rows, path)
    print(f"[cache] saved {path}: {len(rows)} rows, skipped={skipped}")

    if len(rows) == 0:
        raise ValueError("Cache is empty. Check GSM8K parquet fields.")

    return rows


class GSM8KReasoningDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "reason": r["reason"],
            "reason_mask": r["reason_mask"],
            "y_raw": torch.tensor(r["y_raw"], dtype=torch.float32),
            "y_log": torch.tensor(r["y_log"], dtype=torch.float32),
        }


def collate(batch):
    return {
        "q": torch.stack([b["q"] for b in batch]),
        "qmask": torch.stack([b["qmask"] for b in batch]),
        "reason": torch.stack([b["reason"] for b in batch]),
        "reason_mask": torch.stack([b["reason_mask"] for b in batch]),
        "y_raw": torch.stack([b["y_raw"] for b in batch]),
        "y_log": torch.stack([b["y_log"] for b in batch]),
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm(cfg: Config):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: Config, device):
    model = build_hlcm(cfg).to(device)

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print("[load] H-LCM:", cfg.ckpt_path)
    print("[load] missing:", len(missing))
    print("[load] unexpected:", len(unexpected))

    return model


class MathAnswerHead(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class HLCMMathReasoner(nn.Module):
    def __init__(self, cfg: Config, hlcm, mu=None, sigma=None):
        super().__init__()
        self.hlcm = hlcm
        self.mu = mu
        self.sigma = sigma
        self.answer_head = MathAnswerHead(cfg.model_dim, cfg.head_dropout)

    def encode(self, x, mask):
        device = next(self.parameters()).device

        x = x.to(device)
        mask = mask.to(device)

        if self.mu is not None and self.sigma is not None:
            x = (x - self.mu.view(1, 1, -1)) / self.sigma.view(1, 1, -1)

        h = self.hlcm(x)
        h_tan = self.hlcm.manifold.logmap0(h)

        B, T, D = h_tan.shape
        out = torch.empty(B, D, device=h_tan.device, dtype=h_tan.dtype)

        for i in range(B):
            valid = (~mask[i]).nonzero(as_tuple=False).view(-1)
            j = int(valid[-1].item()) if valid.numel() else 0
            out[i] = h_tan[i, j]

        return F.normalize(out, dim=-1)

    def forward(self, batch):
        q_emb = self.encode(batch["q"], batch["qmask"])
        pred_log = self.answer_head(q_emb)
        return pred_log, q_emb

    @torch.no_grad()
    def encode_reasoning_target(self, batch):
        return self.encode(batch["reason"], batch["reason_mask"])


def configure_trainable_params(model: HLCMMathReasoner, cfg: Config):
    if cfg.freeze_hlcm:
        set_requires_grad(model.hlcm, False)
        set_requires_grad(model.answer_head, True)
        return

    set_requires_grad(model.hlcm, False)
    set_requires_grad(model.answer_head, True)

    if not hasattr(model.hlcm, "layers"):
        raise AttributeError("HyperbolicLCM must have attribute `layers`.")

    layers = list(model.hlcm.layers)
    n = max(1, min(cfg.n_last_blocks_unfrozen, len(layers)))

    for block in layers[-n:]:
        set_requires_grad(block, True)


# ============================================================
# LOSS / EVAL
# ============================================================

def reasoning_contrastive_loss(q_emb, r_emb, temperature):
    q = F.normalize(q_emb, dim=-1)
    r = F.normalize(r_emb, dim=-1)

    logits = q @ r.t()
    logits = logits / temperature

    labels = torch.arange(q.size(0), device=q.device)

    return F.cross_entropy(logits, labels)


def compute_loss(model, batch, cfg, device):
    pred_log, q_emb = model(batch)

    y_log = batch["y_log"].to(device)

    answer_loss = F.smooth_l1_loss(pred_log, y_log)

    with torch.no_grad():
        r_emb = model.encode_reasoning_target(batch)

    reason_loss = reasoning_contrastive_loss(
        q_emb=q_emb,
        r_emb=r_emb,
        temperature=cfg.contrastive_temperature,
    )

    loss = (
        cfg.answer_loss_weight * answer_loss
        + cfg.reasoning_loss_weight * reason_loss
    )

    return loss, answer_loss, reason_loss, pred_log


@torch.no_grad()
def evaluate(model, loader, cfg, device):
    model.eval()

    total_loss = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_exact = 0.0
    total_n = 0

    use_amp = cfg.use_bf16 and device.type == "cuda"

    for batch in loader:
        if device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=use_amp,
                dtype=build_amp_dtype(device),
            ):
                loss, answer_loss, reason_loss, pred_log = compute_loss(model, batch, cfg, device)
        else:
            loss, answer_loss, reason_loss, pred_log = compute_loss(model, batch, cfg, device)

        y_raw = batch["y_raw"].to(device)
        pred_raw = log_value_to_answer(pred_log.float())

        bs = y_raw.size(0)

        total_loss += float(loss.item()) * bs
        total_mae += torch.sum(torch.abs(pred_raw - y_raw)).item()
        total_rmse += torch.sum((pred_raw - y_raw) ** 2).item()
        total_exact += torch.sum(torch.round(pred_raw) == torch.round(y_raw)).item()
        total_n += bs

    return {
        "loss": total_loss / max(1, total_n),
        "mae": total_mae / max(1, total_n),
        "rmse": math.sqrt(total_rmse / max(1, total_n)),
        "rounded_acc": total_exact / max(1, total_n),
    }


def make_optimizer_scheduler(model, cfg, total_steps):
    head_params = [p for p in model.answer_head.parameters() if p.requires_grad]
    backbone_params = [p for p in model.hlcm.parameters() if p.requires_grad]

    groups = [{"params": head_params, "lr": cfg.head_lr}]

    if len(backbone_params) > 0:
        groups.append({"params": backbone_params, "lr": cfg.backbone_lr})

    opt = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)

    warmup = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


# ============================================================
# TRAIN
# ============================================================

def train(cfg: Config):
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)

    if cfg.rebuild_cache:
        shutil.rmtree(cfg.cache_dir, ignore_errors=True)
        ensure_dir(cfg.cache_dir)

    conceptizer = DebertaConceptizer(cfg)

    train_hf, test_hf = load_gsm8k(cfg)

    train_rows = build_or_load_cache(cfg, "train", train_hf, conceptizer)
    test_rows = build_or_load_cache(cfg, "test", test_hf, conceptizer)

    del conceptizer
    cuda_cleanup()

    train_ds = GSM8KReasoningDataset(train_rows)
    test_ds = GSM8KReasoningDataset(test_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        collate_fn=collate,
        pin_memory=(device.type == "cuda"),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        collate_fn=collate,
        pin_memory=(device.type == "cuda"),
    )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    hlcm = load_pretrained_hlcm(cfg, device)
    model = HLCMMathReasoner(cfg, hlcm, mu=mu, sigma=sigma).to(device)

    configure_trainable_params(model, cfg)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum_steps)
    total_steps = max(1, steps_per_epoch * cfg.epochs)

    opt, sched = make_optimizer_scheduler(model, cfg, total_steps)

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and cfg.use_bf16 and not torch.cuda.is_bf16_supported()),
    )

    best_acc = -1.0
    best_state = clone_state_dict_to_cpu(model)

    train_log = os.path.join(cfg.out_dir, "train_log.csv")
    eval_log = os.path.join(cfg.out_dir, "eval_log.csv")

    use_amp = cfg.use_bf16 and device.type == "cuda"
    global_step = 0
    t0 = time.time()

    base = evaluate(model, test_loader, cfg, device)
    print("[BASE]", base)

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        running_loss = 0.0
        running_answer_loss = 0.0
        running_reason_loss = 0.0
        running_n = 0

        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}", dynamic_ncols=True)

        for step, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, ans_loss, rea_loss, pred_log = compute_loss(model, batch, cfg, device)
                    loss_backward = loss / cfg.grad_accum_steps
            else:
                loss, ans_loss, rea_loss, pred_log = compute_loss(model, batch, cfg, device)
                loss_backward = loss / cfg.grad_accum_steps

            bs = batch["y_log"].size(0)

            running_loss += float(loss.item()) * bs
            running_answer_loss += float(ans_loss.item()) * bs
            running_reason_loss += float(rea_loss.item()) * bs
            running_n += bs

            if scaler.is_enabled():
                scaler.scale(loss_backward).backward()
            else:
                loss_backward.backward()

            do_step = (step % cfg.grad_accum_steps == 0) or (step == len(train_loader))

            if do_step:
                trainable = [p for p in model.parameters() if p.requires_grad]

                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_step += 1

            pbar.set_postfix(
                loss=f"{running_loss / max(1, running_n):.4f}",
                ans=f"{running_answer_loss / max(1, running_n):.4f}",
                reason=f"{running_reason_loss / max(1, running_n):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        eval_result = evaluate(model, test_loader, cfg, device)
        print(f"[epoch {epoch}] {eval_result}")

        row = {
            "epoch": epoch,
            "global_step": global_step,
            "train_loss": running_loss / max(1, running_n),
            "train_answer_loss": running_answer_loss / max(1, running_n),
            "train_reasoning_loss": running_reason_loss / max(1, running_n),
            "eval_loss": eval_result["loss"],
            "eval_mae": eval_result["mae"],
            "eval_rmse": eval_result["rmse"],
            "eval_rounded_acc": eval_result["rounded_acc"],
        }

        file_exists = os.path.exists(eval_log)
        with open(eval_log, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if not file_exists:
                writer.writeheader()
            writer.writerow(row)

        if eval_result["rounded_acc"] > best_acc:
            best_acc = eval_result["rounded_acc"]
            best_state = clone_state_dict_to_cpu(model)

            torch.save(
                {
                    "model": best_state,
                    "epoch": epoch,
                    "best_rounded_acc": best_acc,
                    "config": cfg.__dict__,
                },
                os.path.join(cfg.out_dir, "best.pt"),
            )

            print("[save] best.pt")

        torch.save(
            {
                "model": clone_state_dict_to_cpu(model),
                "epoch": epoch,
                "config": cfg.__dict__,
            },
            os.path.join(cfg.out_dir, "last.pt"),
        )

        cuda_cleanup()

    model.load_state_dict(best_state, strict=True)
    final = evaluate(model, test_loader, cfg, device)

    summary = {
        "dataset": cfg.dataset_name,
        "final_test_loss": final["loss"],
        "final_test_mae": final["mae"],
        "final_test_rmse": final["rmse"],
        "final_test_rounded_acc": final["rounded_acc"],
        "best_test_rounded_acc": best_acc,
        "total_minutes": (time.time() - t0) / 60.0,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "epochs": cfg.epochs,
        "answer_loss_weight": cfg.answer_loss_weight,
        "reasoning_loss_weight": cfg.reasoning_loss_weight,
    }

    with open(os.path.join(cfg.out_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n========== FINAL ==========")
    print(summary)


if __name__ == "__main__":
    cfg = Config()
    train(cfg)

Device: cuda:0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[GSM8K] train: 7473
[GSM8K] test: 1319
[GSM8K] columns: ['question', 'answer']
[GSM8K] first example: {'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?', 'answer': 'Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16'}


cache:train: 100%|██████████████████████████████████████████████| 7473/7473 [07:50<00:00, 15.88it/s]


...

[epoch 1] {'loss': 2.282487476658333, 'mae': 6955.2331611916725, 'rmse': 92005.73178721314, 'rounded_acc': 0.0}
[save] best.pt


epoch 2/3: 100%|█| 3737/3737 [16:29<00:00,  3.78it/s, ans=1.1811, loss=1.3894, lr=1.32e-05, reason=0


[epoch 2] {'loss': 1.621957043158999, 'mae': 6817.912515202103, 'rmse': 92044.14510963568, 'rounded_acc': 0.002274450341167551}
[save] best.pt


epoch 3/3: 100%|█| 3737/3737 [16:46<00:00,  3.71it/s, ans=1.1686, loss=1.3763, lr=0.00e+00, reason=0


[epoch 3] {'loss': 1.7258753819028205, 'mae': 6828.533701089045, 'rmse': 92037.48495030546, 'rounded_acc': 0.001516300227445034}

========== FINAL ==========
{'dataset': 'GSM8K', 'final_test_loss': 1.621957043158999, 'final_test_mae': 6817.912515202103, 'final_test_rmse': 92044.14510963568, 'final_test_rounded_acc': 0.002274450341167551, 'best_test_rounded_acc': 0.002274450341167551, 'total_minutes': 56.81880614360173, 'total_params': 2436501506, 'trainable_params': 419553282, 'epochs': 3, 'answer_loss_weight': 1.0, 'reasoning_loss_weight': 0.3}
